# ALQAC 2026 — retrieval top-14 (file) + Qwen3.5-9B robust ensemble v9

Pipeline ưu tiên đồng thời coverage và chất lượng:

1. Issue Mapper rất ngắn, chỉ inventory các relief có trong CASE_QUERY.
2. Ba Legal Judge votes độc lập theo sampling profile chính thức của Qwen3 non-thinking.
3. Majority vote tạo prediction; direct fallback và hard fallback bảo đảm luôn có đủ 50 dòng submission.

Mọi structured output là compact JSON với disable_any_whitespace=True để chặn whitespace loop. Output không chứa reasoning tự do dài; các trường quyết định đều là enum.

Thiết lập Modal Secret: HF_TOKEN. Upload ALQAC2026_public_test.json, agent_v4_results.json và retrieval_top14_ours.json (điều luật đã truy xuất sẵn cho từng case — notebook không còn search Qdrant; đổi file qua ALQAC_RETRIEVAL_PATH). Để paired comparison chạy được, upload predictions_qwen3-4b_seed-2026.json hoặc đặt ALQAC_BASE_PREDICTIONS_PATH.


In [1]:
%%capture
# LLM chay TU XA tren Modal GPU => notebook chi can client + retrieval (khong cai vLLM/torch nang o day).
# Chay 1 lan, KHONG can restart kernel -> Run All chay mot mach.
%pip install -q modal sentence-transformers 'transformers==5.14.1' python-dotenv pandas scikit-learn tqdm numpy


In [2]:
import ast
import gc
import hashlib
import json
import platform
import os
import random
import re
import time
from importlib.metadata import version as package_version
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from dotenv import find_dotenv, load_dotenv
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

# Chay local + offload LLM sang Modal GPU. Can: MODAL_TOKEN_ID/MODAL_TOKEN_SECRET (Modal CLI hoac .env),
# HF_TOKEN (tai model o remote). Retrieval KHONG dung Qdrant nua: doc dieu luat tu file da chuan bi san.
env_path = find_dotenv(usecwd=True)
if not env_path:
    for candidate in [Path.cwd() / '.env', Path.cwd().parent / '.env', Path('/root/.env'), Path('../.env')]:
        if candidate.exists():
            env_path = str(candidate.resolve())
            break
if env_path:
    load_dotenv(env_path, override=False)
    print('Loaded .env:', env_path)
else:
    print('Khong tim thay .env; doc MODAL_TOKEN_ID/MODAL_TOKEN_SECRET/HF_TOKEN tu environment.')
# Tren Modal Notebooks: KHONG can MODAL_TOKEN_* (kernel da xac thuc san),
# va HF_TOKEN do Modal Secret bom vao environment.
for _k in ['HF_TOKEN']:
    print(' ', _k, 'OK' if os.environ.get(_k) else 'MISSING -> gan Modal Secret hoac dat bien moi truong')
for _k in ['MODAL_TOKEN_ID', 'MODAL_TOKEN_SECRET']:
    print(' ', _k, 'OK' if os.environ.get(_k) else 'trong (BINH THUONG neu chay ngay tren Modal Notebooks)')

# Goc project (noi co .env) -> dung de tim dataset & .env cho Modal, khong phu thuoc cwd cua kernel.
PROJECT_ROOT = Path(env_path).parent if env_path else Path.cwd()
print('PROJECT_ROOT:', PROJECT_ROOT)

# Mỗi notebook chỉ load một model đầy đủ lên GPU.
AVAILABLE_MODELS = ['Qwen3.5-9B']
MODEL_REPOS = {
    'Llama-3.1-8B-Instruct': 'meta-llama/Llama-3.1-8B-Instruct',
    'Qwen3-4B': 'Qwen/Qwen3-4B',
    'Qwen2.5-7B-Instruct': 'Qwen/Qwen2.5-7B-Instruct',
    'DeepSeek-R1-Distill-Qwen-1.5B': 'deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B',
    'DeepSeek-R1-Distill-Llama-8B': 'deepseek-ai/DeepSeek-R1-Distill-Llama-8B',
    'Llama-3.2-3B': 'meta-llama/Llama-3.2-3B-Instruct',
    'Qwen3.5-9B': 'Qwen/Qwen3.5-9B',
}

# Chỉ decoding profile được phép khác nhau theo khuyến nghị của nhà sản xuất.
# Mọi retrieval, prompt content, token budget, seed và metric ở dưới đều giống nhau.
MODEL_GENERATION_PROFILES = {
    'Llama-3.1-8B-Instruct': {
        'profile_name': 'vendor_model_generation_config',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': True,
        'generation_kwargs': {},
    },
    'Llama-3.2-3B': {
        'profile_name': 'vendor_model_generation_config',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': True,
        'generation_kwargs': {},
    },
    'Qwen2.5-7B-Instruct': {
        'profile_name': 'vendor_qwen2_5_instruct',
        'enable_thinking': False,
        'use_system_prompt': True,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.7, 'top_p': 0.8,
            'top_k': 20, 'repetition_penalty': 1.05,
        },
    },
    'Qwen3-4B': {
        'profile_name': 'vendor_qwen3_thinking',
        'enable_thinking': True,
        'use_system_prompt': True,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95, 'top_k': 20,
        },
    },
    'DeepSeek-R1-Distill-Qwen-1.5B': {
        'profile_name': 'vendor_deepseek_r1_distill',
        'enable_thinking': True,
        'use_system_prompt': False,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95,
        },
    },
    'DeepSeek-R1-Distill-Llama-8B': {
        'profile_name': 'vendor_deepseek_r1_distill',
        'enable_thinking': True,
        'use_system_prompt': False,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 0.6, 'top_p': 0.95,
        },
    },
    'Qwen3.5-9B': {
        'profile_name': 'vendor_qwen3_5_thinking_general',
        'enable_thinking': True,
        'use_system_prompt': True,
        'use_model_generation_config': False,
        'generation_kwargs': {
            'do_sample': True, 'temperature': 1.0, 'top_p': 0.95,
            'top_k': 20, 'min_p': 0.0, 'repetition_penalty': 1.0,
        },
    },
}

assert len(AVAILABLE_MODELS) == 1, 'Mỗi lần chỉ load một model đầy đủ lên GPU.'
MODEL_NAME = AVAILABLE_MODELS[0]
MODEL_ID = MODEL_REPOS[MODEL_NAME]
# Revision theo từng model. Base tự lấy từ model đã load; Modal phải chỉ định rõ,
# nên đổi model mà quên đổi revision sẽ gây 404 RevisionNotFound.
MODEL_REVISIONS = {
    'Qwen3-4B': '1cfa9a7208912126459214e8b04321603b3df60c',
    'Qwen2.5-7B-Instruct': 'main',
    'Llama-3.1-8B-Instruct': 'main',
    'Llama-3.2-3B': 'main',
    'DeepSeek-R1-Distill-Qwen-1.5B': 'main',
    'DeepSeek-R1-Distill-Llama-8B': 'main',
    'Qwen3.5-9B': 'main',
}
MODEL_REVISION = os.getenv('MODEL_REVISION', MODEL_REVISIONS.get(MODEL_NAME, 'main'))
MODELS_TO_RUN = [MODEL_NAME]
ACTIVE_PROFILE = MODEL_GENERATION_PROFILES[MODEL_NAME]

BENCHMARK_VERSION = 'v12_legal_pado_divisible_types_balanced_judge_private'
BENCHMARK_PROTOCOL = 'legal_pado_thinking_divisible_mapper_3vote_derived_verdict'
# ---- Retrieval: dung file dieu luat da chuan bi san, KHONG search Qdrant ----
# Doi nguon retrieval => doi retrieval_signature => doi pipeline_contract_hash va cache_key,
# nen ket qua cu tu dong bi tinh lai chu khong bi dung nham.
RETRIEVAL_FILE_NAME = os.getenv('ALQAC_RETRIEVAL_FILE_NAME', 'retrieval_top14_private.json')
RETRIEVAL_SOURCE = f'file:{RETRIEVAL_FILE_NAME}'
TOP_K = int(os.getenv('ALQAC_TOP_K', '14'))
MAX_INPUT_TOKENS = 24000
PADO_PIPELINE_VERSION = 'legal-pado-v15-private-submission'
# Thinking mode (khớp base) cần token cho reasoning + JSON; ceiling theo base = 8192.
MAX_NEW_TOKENS_ISSUE = 4096
MAX_NEW_TOKENS_JUDGE = 16384
MAX_NEW_TOKENS_DIRECT = 2048
JUDGE_VOTES = 3
MAX_NEW_TOKENS = max(MAX_NEW_TOKENS_ISSUE, MAX_NEW_TOKENS_JUDGE, MAX_NEW_TOKENS_DIRECT)

# ---- vLLM engine knobs ----
ENABLE_THINKING = ACTIVE_PROFILE['enable_thinking']  # theo profile của model (đổi model là tự khớp)
USE_STRUCTURED_OUTPUTS = False  # Base sinh JSON tự do + parse robust; grammar sẽ đổi phân phối decoding.
VLLM_DTYPE = os.getenv('VLLM_DTYPE', 'bfloat16')  # khớp base; dùng L4/A10/A100
assert VLLM_DTYPE == 'bfloat16', 'Base đã chạy BF16; đổi dtype sẽ làm mất tính so sánh.'
GPU_MEM_UTIL = float(os.getenv('VLLM_GPU_MEM_UTIL', '0.92'))
# Top-14 dai hon top-10 ~2.6x: prompt judge worst-case ~16.3k token, cong max_new 16384 thi
# vuot 32768. Qwen3 co max_position_embeddings=40960 nen nang tran thay vi cat noi dung luat.
# Neu engine tu choi 40960: dat VLLM_MAX_MODEL_LEN=32768 va ALQAC_MAX_ARTICLE_CHARS=4000.
MAX_MODEL_LEN = int(os.getenv('VLLM_MAX_MODEL_LEN', '40960'))
MAX_NUM_SEQS = int(os.getenv('VLLM_MAX_NUM_SEQS', '64'))
VLLM_ENFORCE_EAGER = os.getenv('VLLM_ENFORCE_EAGER', '0') == '1'
VLLM_RUNTIME_VERSION = '0.25.1'
# Qwen3 thinking vendor profile (khớp base); seed riêng cho từng vote để reproducible.
SAMPLING = {k: v for k, v in ACTIVE_PROFILE['generation_kwargs'].items() if k != 'do_sample'}
MAX_ARTICLE_CHARS = int(os.getenv('ALQAC_MAX_ARTICLE_CHARS', '6000'))  # giới hạn theo từng điều; mọi model nhận cùng chuỗi evidence
MAX_GENERATION_ATTEMPTS = 1  # strict: mỗi stage chỉ được gọi đúng một lần
FAIL_FAST = False
INVALID_OUTPUT_LABEL = '__INVALID_OUTPUT__'
EVAL_SEEDS = [2026]
OUTPUT_DIR = Path('outputs_alqac_e2e') / BENCHMARK_VERSION
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = ['A_WIN', 'B_WIN', 'PARTIAL_A_WIN', 'PARTIAL_B_WIN']
assert MAX_GENERATION_ATTEMPTS == 1
assert len(EVAL_SEEDS) == len(set(EVAL_SEEDS))
random.seed(EVAL_SEEDS[0])
np.random.seed(EVAL_SEEDS[0])

def get_secret(*names, required=True):
    for name in names:
        value = os.getenv(name)
        if value:
            return value
    if required:
        raise RuntimeError(
            f'Thiếu secret {names}. Hãy gắn Modal Secret vào notebook hoặc khai báo biến môi trường tương ứng.'
        )
    return None

HF_TOKEN = get_secret('HF_TOKEN', required=False)

# LLM chay tren Modal GPU (remote). Notebook chi can CPU: khong con buoc embedding local.
torch.manual_seed(EVAL_SEEDS[0])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(EVAL_SEEDS[0])
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Retrieval:', RETRIEVAL_SOURCE, '| top_k:', TOP_K, '| local torch:', torch.__version__)
print('Model (remote vLLM):', MODEL_NAME, '->', MODEL_ID)
print('Benchmark:', BENCHMARK_VERSION, '| protocol:', BENCHMARK_PROTOCOL)
print('Generation profile:', ACTIVE_PROFILE['profile_name'], '| seeds:', EVAL_SEEDS)
print('vLLM dtype:', VLLM_DTYPE, '| unquantized')


Khong tim thay .env; doc MODAL_TOKEN_ID/MODAL_TOKEN_SECRET/HF_TOKEN tu environment.
  HF_TOKEN OK
  MODAL_TOKEN_ID OK
  MODAL_TOKEN_SECRET OK
PROJECT_ROOT: /root
Retrieval: file:retrieval_top14_private.json | top_k: 14 | local torch: 2.8.0+cu129
Model (remote vLLM): Qwen3.5-9B -> Qwen/Qwen3.5-9B
Benchmark: v12_legal_pado_divisible_types_balanced_judge_private | protocol: legal_pado_thinking_divisible_mapper_3vote_derived_verdict
Generation profile: vendor_qwen3_5_thinking_general | seeds: [2026]
vLLM dtype: bfloat16 | unquantized


In [3]:
# PRIVATE TEST (bai thi): chi co case_id + case_query, KHONG co verdict_label.
# Moi assert/nhanh lien quan den nhan da duoc bo o duoi.
PRIVATE_TEST_FILE = os.getenv('ALQAC_PRIVATE_TEST_FILE', 'ALQAC_private_test.json')


def find_public_test():
    # Có thể override mà không sửa notebook: ALQAC_PRIVATE_TEST_PATH=/path/to/file.json
    candidates = []
    for _env in ('ALQAC_PRIVATE_TEST_PATH', 'ALQAC_PUBLIC_TEST_PATH'):
        if os.getenv(_env):
            candidates.append(Path(os.environ[_env]))
    candidates += [
        PROJECT_ROOT / 'data' / PRIVATE_TEST_FILE,
        PROJECT_ROOT / PRIVATE_TEST_FILE,
        Path.cwd() / PRIVATE_TEST_FILE,
        Path('/root') / PRIVATE_TEST_FILE,
        Path('/kaggle/input/datasets/ldhhieu18/demnguoctoibinhminh') / PRIVATE_TEST_FILE,
        Path('data') / PRIVATE_TEST_FILE,
        Path('../data') / PRIVATE_TEST_FILE,
        Path('/kaggle/working') / PRIVATE_TEST_FILE,
    ]
    # --- Modal Notebooks: file ban upload thuong nam canh notebook (cwd) hoac trong home/root. ---
    candidates += [
        Path.home() / PRIVATE_TEST_FILE,
        Path('/root/data') / PRIVATE_TEST_FILE,
        Path('/workspace') / PRIVATE_TEST_FILE,
        Path('/notebooks') / PRIVATE_TEST_FILE,
    ]
    for _root in {Path.cwd(), Path.home(), Path('/root')}:
        try:
            if _root.exists():
                candidates.extend(sorted(_root.rglob(PRIVATE_TEST_FILE))[:5])
        except (PermissionError, OSError):
            pass
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob(PRIVATE_TEST_FILE))
    for path in candidates:
        if path.is_file():
            return path.resolve()
    checked = '\n'.join(f'  - {path}' for path in candidates)
    raise FileNotFoundError(f'Không tìm thấy {PRIVATE_TEST_FILE}. Đã kiểm tra:\n{checked}')


DATA_PATH = find_public_test()
with DATA_PATH.open(encoding='utf-8') as f:
    public_data = json.load(f)

N_CASES = len(public_data)
assert N_CASES > 0, 'File private test rong'
assert len({x['case_id'] for x in public_data}) == N_CASES, 'Trung case_id'
assert all(x.get('case_query') for x in public_data), 'Co case thieu case_query'
# KHONG assert verdict_label: private test la bai thi, khong kem nhan.
HAS_GOLD = all(x.get('verdict_label') in LABELS for x in public_data)
print('Private test:', N_CASES, 'case | co nhan gold:', HAS_GOLD)

# Đây là view duy nhất được pipeline dự đoán sử dụng. Gold được giữ riêng cho cell đánh giá.
# ---- Nap evidence bo sung (agent_v4_private_results.json) ----
AGENT_EVIDENCE_FILE = os.getenv('ALQAC_AGENT_EVIDENCE_FILE', 'agent_v4_private_results.json')


def find_agent_evidence():
    candidates = []
    if os.getenv('ALQAC_AGENT_EVIDENCE_PATH'):
        candidates.append(Path(os.environ['ALQAC_AGENT_EVIDENCE_PATH']))
    for base in [PROJECT_ROOT / 'data', PROJECT_ROOT, Path.cwd(), Path.home(),
                 Path('/root'), Path('/root/data'), Path('/workspace'), Path('/notebooks')]:
        candidates.append(base / AGENT_EVIDENCE_FILE)
    for _root in {Path.cwd(), Path.home(), Path('/root')}:
        try:
            if _root.exists():
                candidates.extend(sorted(_root.rglob(AGENT_EVIDENCE_FILE))[:5])
        except (PermissionError, OSError):
            pass
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob(AGENT_EVIDENCE_FILE))
    for path in candidates:
        if path.is_file():
            return path.resolve()
    raise FileNotFoundError(f'Khong tim thay {AGENT_EVIDENCE_FILE} (evidence bo sung).')

AGENT_EVIDENCE_PATH = find_agent_evidence()
with AGENT_EVIDENCE_PATH.open(encoding='utf-8') as f:
    agent_evidence_raw = json.load(f)

# Gioi han so ky tu evidence dua vao prompt (tranh vuot MAX_INPUT_TOKENS). Chinh qua env neu can.
MAX_EVIDENCE_CHARS = int(os.getenv('ALQAC_MAX_EVIDENCE_CHARS', '8000'))

def _case_evidence_text(record, max_chars=MAX_EVIDENCE_CHARS):
    seen, parts, total = set(), [], 0
    for e in record.get('evidence_details', []):
        chunk_id = e.get('chunk_id')
        txt = str(e.get('text', '')).strip()
        if not txt or chunk_id in seen:
            continue
        seen.add(chunk_id)
        if total + len(txt) + 1 > max_chars:
            break
        parts.append(txt)
        total += len(txt) + 1
    return (chr(10)).join(parts)

evidence_by_case = {r['case_id']: _case_evidence_text(r) for r in agent_evidence_raw}


def _case_evidence_ids(record):
    """ID cac segment evidence cua case, GIU NGUYEN thu tu, bo trung.

    Submission yeu cau truong 'case_evidence' la danh sach id segment (vd
    'case_4101_seg_...'). Uu tien khoa 'evidence' cua agent_v4; neu thieu thi lay
    chunk_id trong 'evidence_details'.
    """
    ids, seen = [], set()
    for value in (record.get('evidence') or []):
        sid = str(value).strip()
        if sid and sid not in seen:
            seen.add(sid)
            ids.append(sid)
    if not ids:
        for detail in (record.get('evidence_details') or []):
            sid = str(detail.get('chunk_id', '')).strip()
            if sid and sid not in seen:
                seen.add(sid)
                ids.append(sid)
    return ids


evidence_ids_by_case = {r['case_id']: _case_evidence_ids(r) for r in agent_evidence_raw}
_missing_ev = [x['case_id'] for x in public_data if not evidence_by_case.get(x['case_id'])]
if _missing_ev:
    print('CANH BAO: thieu evidence cho case:', _missing_ev)
_n_ev_ids = sum(1 for x in public_data if evidence_ids_by_case.get(x['case_id']))
print('Evidence bo sung:', AGENT_EVIDENCE_PATH,
      '| co text:', N_CASES - len(_missing_ev), '/', N_CASES,
      '| co id segment:', _n_ev_ids, '/', N_CASES)

inference_cases = [
    {
        'case_id': x['case_id'],
        'case_query': x['case_query'],
        'case_facts': evidence_by_case.get(x['case_id'], ''),
    }
    for x in public_data
]
# Private test khong kem nhan -> gold rong; cell danh gia da chuyen sang bao cao suc khoe.
gold_by_case = {x['case_id']: x.get('verdict_label') for x in public_data} if HAS_GOLD else {}

print('Dataset:', DATA_PATH)
print('Cases:', len(inference_cases))
print('Retrieval: nap tu file o cell sau, khong ket noi Qdrant va khong load BGE-M3.')


Private test: 60 case | co nhan gold: False
Evidence bo sung: /root/agent_v4_private_results.json | co text: 60 / 60 | co id segment: 60 / 60
Dataset: /root/ALQAC_private_test.json
Cases: 60
Retrieval: nap tu file o cell sau, khong ket noi Qdrant va khong load BGE-M3.


In [4]:
# ==== Retrieval: NAP TU FILE, khong search Qdrant ====
# Truoc day cell nay encode case_query bang BGE-M3 roi query Qdrant lay top-10.
# Gio dieu luat cho tung case da duoc chuan bi san ngoai notebook. Cell chi doc, kiem tra
# va do vao retrieval_cache theo DUNG schema cu (rank/score/law_id/aid/article_no/content_Article)
# nen toan bo cac cell phia sau khong phai doi gi.
def find_retrieval_file():
    candidates = []
    if os.getenv('ALQAC_RETRIEVAL_PATH'):
        candidates.append(Path(os.environ['ALQAC_RETRIEVAL_PATH']))
    for base in [PROJECT_ROOT / 'data', PROJECT_ROOT, Path.cwd(), Path.cwd() / 'data',
                 Path.home(), Path('/root'), Path('/root/data'), Path('/workspace'), Path('/notebooks')]:
        candidates.append(base / RETRIEVAL_FILE_NAME)
    for _root in {Path.cwd(), Path.home(), Path('/root')}:
        try:
            if _root.exists():
                candidates.extend(sorted(_root.rglob(RETRIEVAL_FILE_NAME))[:5])
        except (PermissionError, OSError):
            pass
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob(RETRIEVAL_FILE_NAME))
    for path in candidates:
        if path.is_file():
            return path.resolve()
    raise FileNotFoundError(
        f'Khong tim thay {RETRIEVAL_FILE_NAME}. Dat file canh notebook, vao thu muc data/, '
        f'hoac tro duong dan bang ALQAC_RETRIEVAL_PATH.'
    )


LAW_FIELDS = ('rank', 'score', 'law_id', 'aid', 'article_no', 'content_Article')


def normalize_laws(case_id, items):
    """Giu nguyen thu tu trong file, bo trung (law_id, aid), danh lai rank 1..N."""
    if not isinstance(items, list):
        raise ValueError(f'{case_id}: gia tri phai la list dieu luat, nhan duoc {type(items).__name__}')
    laws, seen = [], set()
    for item in items:
        missing = [f for f in LAW_FIELDS if f not in item]
        if missing:
            raise ValueError(f'{case_id}: mot dieu luat thieu truong {missing}')
        key = (str(item['law_id']), int(item['aid']))
        if key in seen:
            continue
        seen.add(key)
        laws.append({
            'rank': len(laws) + 1,
            'score': float(item['score']),
            'law_id': key[0],
            'aid': key[1],
            'article_no': int(item['article_no']),
            'content_Article': str(item['content_Article'] or ''),
        })
    if not laws:
        raise ValueError(f'{case_id}: khong co dieu luat nao')
    return laws


RETRIEVAL_PATH = find_retrieval_file()
_raw_retrieval = json.loads(RETRIEVAL_PATH.read_text(encoding='utf-8'))
assert isinstance(_raw_retrieval, dict), 'File retrieval phai la dict {case_id: [dieu luat, ...]}'

expected_case_ids = {x['case_id'] for x in inference_cases}
_missing_cases = sorted(expected_case_ids - set(_raw_retrieval))
assert not _missing_cases, f'File retrieval thieu {len(_missing_cases)} case: {_missing_cases[:5]}'

retrieval_cache = {cid: normalize_laws(cid, _raw_retrieval[cid]) for cid in expected_case_ids}
_sizes = {len(v) for v in retrieval_cache.values()}
assert _sizes == {TOP_K}, f'So dieu luat/case = {sorted(_sizes)}, khong khop TOP_K={TOP_K}'
assert all(law['content_Article'].strip()
           for laws in retrieval_cache.values() for law in laws), 'Co dieu luat rong content_Article'

# Chu ky nay di vao manifest + pipeline contract + cache_key: doi file la moi case tu dong chay lai.
retrieval_signature = hashlib.sha256(
    json.dumps(
        {'source': RETRIEVAL_SOURCE, 'top_k': TOP_K, 'laws': retrieval_cache},
        ensure_ascii=False, sort_keys=True,
    ).encode('utf-8')
).hexdigest()

sample_case = inference_cases[0]
sample_laws = retrieval_cache[sample_case['case_id']]
display(pd.DataFrame(sample_laws)[['rank', 'score', 'law_id', 'article_no', 'aid']])

_law_chars = [sum(len(law['content_Article'][:MAX_ARTICLE_CHARS]) for law in laws)
              for laws in retrieval_cache.values()]
print('Retrieval file:', RETRIEVAL_PATH)
print('Cases:', len(retrieval_cache), '| dieu luat/case:', TOP_K,
      '| signature:', retrieval_signature[:12])
print('Law context sau khi cat', MAX_ARTICLE_CHARS, 'ky tu/dieu: TB',
      round(sum(_law_chars) / len(_law_chars)), '| MAX', max(_law_chars), 'ky tu')


,rank,score,law_id,article_no,aid
0,1,0.031025,45/2013/QH13,188,56138
1,2,0.031010,45/2013/QH13,167,56117
2,3,0.030282,45/2013/QH13,168,56118
3,4,0.029828,43/2014/NĐ-CP,82,9577
4,5,0.028629,43/2014/NĐ-CP,91,9586
5,6,0.028205,43/2014/NĐ-CP,81,9576
6,7,0.023399,92/2015/QH13,26,50691
7,8,0.500000,92/2015/QH13,147,50812
8,9,0.500000,92/2015/QH13,35,50700
9,10,0.500000,92/2015/QH13,39,50704


Retrieval file: /root/retrieval_top14_private.json
Cases: 60 | dieu luat/case: 14 | signature: 3cdf4fc5911f
Law context sau khi cat 6000 ky tu/dieu: TB 26895 | MAX 39155 ky tu


In [5]:
# ==== Robust compact structured vLLM engine on Modal ====
import modal
import unicodedata

prompt_tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, token=HF_TOKEN, revision=MODEL_REVISION, trust_remote_code=True
)

APP_NAME = 'alqac-e2e-pado-compact-' + MODEL_NAME.lower().replace('.', '-').replace('_', '-')
GPU_TYPE = os.getenv('MODAL_GPU', 'A100-40GB')
app = modal.App(APP_NAME)
MODEL_DTYPE = VLLM_DTYPE

vllm_image = (
    modal.Image.debian_slim(python_version='3.12')
    .pip_install(f'vllm=={VLLM_RUNTIME_VERSION}', 'huggingface_hub')
    .env({'VLLM_USE_FLASHINFER_SAMPLER': '0'})
)
HF_SECRET = [modal.Secret.from_dict({'HF_TOKEN': HF_TOKEN})] if HF_TOKEN else []


@app.cls(image=vllm_image, gpu=GPU_TYPE, timeout=7200, secrets=HF_SECRET)
class RemoteLLM:
    @modal.enter()
    def _load(self):
        from vllm import LLM
        print(f'Loading {MODEL_ID} on Modal {GPU_TYPE}...')
        self.llm = LLM(
            model=MODEL_ID,
            tokenizer=MODEL_ID,
            revision=MODEL_REVISION,
            tokenizer_revision=MODEL_REVISION,
            dtype=MODEL_DTYPE,
            trust_remote_code=True,
            gpu_memory_utilization=GPU_MEM_UTIL,
            max_model_len=MAX_MODEL_LEN,
            max_num_seqs=MAX_NUM_SEQS,
            enforce_eager=VLLM_ENFORCE_EAGER,
            seed=EVAL_SEEDS[0],
        )
        print('vLLM ready on Modal.')

    @modal.method()
    def generate_batch(self, messages_list, seeds, schema, max_tokens, enable_thinking=None):
        from vllm import SamplingParams
        try:
            from vllm.sampling_params import StructuredOutputsParams
        except ImportError:
            from vllm import StructuredOutputsParams

        structured = None
        if USE_STRUCTURED_OUTPUTS and schema is not None:
            structured = StructuredOutputsParams(
                json=schema,
                disable_any_whitespace=True,
            )

        def make_params(seed):
            kwargs = dict(max_tokens=max_tokens, seed=int(seed))
            for _k in ('temperature', 'top_p', 'top_k', 'min_p', 'repetition_penalty'):
                if _k in SAMPLING:
                    kwargs[_k] = SAMPLING[_k]
            if structured is not None:
                kwargs['structured_outputs'] = structured
            return SamplingParams(**kwargs)

        tmpl_kwargs = {}
        if 'qwen3' in MODEL_ID.lower():
            tmpl_kwargs['enable_thinking'] = ENABLE_THINKING if enable_thinking is None else enable_thinking
        outputs = self.llm.chat(
            messages_list,
            [make_params(seed) for seed in seeds],
            add_generation_prompt=True,
            chat_template_kwargs=tmpl_kwargs,
            use_tqdm=True,
        )
        results = []
        for output in outputs:
            generated = output.outputs[0]
            results.append((generated.text, {
                'input_tokens': len(output.prompt_token_ids),
                'output_tokens': len(generated.token_ids),
                'total_tokens': len(output.prompt_token_ids) + len(generated.token_ids),
                'hit_max_new_tokens': generated.finish_reason == 'length',
                'finish_reason': generated.finish_reason,
            }))
        return results


_out_ctx = None
_app_ctx = None
_remote_client = None


def _ensure_remote():
    global _out_ctx, _app_ctx, _remote_client
    if _remote_client is None:
        _out_ctx = modal.enable_output()
        _out_ctx.__enter__()
        _app_ctx = app.run()
        _app_ctx.__enter__()
        _remote_client = RemoteLLM()
    return _remote_client


def close_modal():
    global _out_ctx, _app_ctx, _remote_client
    if _app_ctx is not None:
        _app_ctx.__exit__(None, None, None)
        _app_ctx = None
    if _out_ctx is not None:
        _out_ctx.__exit__(None, None, None)
        _out_ctx = None
    _remote_client = None


MODEL_CONTEXT_LIMIT = MAX_MODEL_LEN
RESOLVED_GENERATION_CONFIG = {
    'engine': 'vllm-modal',
    'structured_outputs': USE_STRUCTURED_OUTPUTS,
    'disable_any_whitespace': USE_STRUCTURED_OUTPUTS,
    'enable_thinking': ENABLE_THINKING,
    'thinking_by_stage': {
        'issue_mapper': False,
        'legal_judge_vote': ENABLE_THINKING,
        'direct_fallback': False,
    },
    'sampling': SAMPLING,
    'judge_votes': JUDGE_VOTES,
    'max_new_tokens_by_stage': {
        'issue_mapper': MAX_NEW_TOKENS_ISSUE,
        'legal_judge_vote': MAX_NEW_TOKENS_JUDGE,
        'direct_fallback': MAX_NEW_TOKENS_DIRECT,
    },
}

LABEL_GUIDE = '''Nhãn cuối (xét theo GIÁ TRỊ phần MATERIAL, không đếm số claim):
- A_WIN: gần như TOÀN BỘ yêu cầu của A được chấp nhận. Chỉ được bỏ qua khác biệt thuần thủ tục (án phí, chi phí tố tụng, làm tròn số lẻ).
  Nếu toà cắt một khoản tiền có giá trị mà A đòi — KỂ CẢ tiền lãi hay tiền phạt — thì đó là PARTIAL_A_WIN chứ KHÔNG phải A_WIN.
- B_WIN: mọi hoặc gần như mọi phần MATERIAL của A bị bác.
- PARTIAL_A_WIN: có ít nhất một phần MATERIAL được chấp nhận VÀ một phần MATERIAL bị bác hoặc bị cắt đáng kể, trong đó A giữ phần lớn giá trị MATERIAL.
- PARTIAL_B_WIN: cũng chia như trên nhưng B giữ phần lớn giá trị MATERIAL.
Chỉ dùng PARTIAL khi chỉ ra được phần MATERIAL cụ thể mà A không đạt được.'''

GROUNDING_RULES = f'''Chỉ dùng CASE_QUERY, THÔNG TIN VỤ VIỆC (evidence, nếu có) và {TOP_K} điều luật truy xuất; A là nguyên đơn, B là bị đơn.
Xem các tình tiết được kể trong CASE_QUERY và THÔNG TIN VỤ VIỆC là dữ kiện, NHƯNG mức tiền/định lượng mà A yêu cầu chỉ là ĐÒI HỎI phải được xét, KHÔNG phải phần toà đã chấp nhận.
Thông tin không xuất hiện là UNKNOWN: không dùng sự im lặng để tự động ACCEPT hoặc REJECT, và không đòi thêm tài liệu chỉ vì bản tóm tắt không liệt kê.
Thực tế xét xử dân sự Việt Nam: toà tách TRÁCH NHIỆM (B có phải bồi thường/thực hiện nghĩa vụ không) khỏi ĐỊNH LƯỢNG (mức tiền cụ thể), và cách xử lý ĐỊNH LƯỢNG phụ thuộc LOẠI yêu cầu:
- Khoản tiền do A tự ước tính (bồi thường thiệt hại, chi phí, tổn thất tinh thần): ngay cả khi trách nhiệm của B rõ, toà RẤT THƯỜNG chỉ chấp nhận MỘT PHẦN (giảm khoản thiếu chứng từ, lỗi hỗn hợp, khấu trừ nghĩa vụ đối ứng).
- Khoản tiền có căn cứ tính toán xác định (nợ gốc theo hợp đồng tín dụng/giấy vay, số tiền bị đơn đã thừa nhận) và các yêu cầu KHÔNG chia nhỏ được (đòi lại đúng thửa đất/tài sản, hủy hoặc công nhận một giao dịch, tiếp tục thực hiện hợp đồng): toà thường chấp nhận TOÀN BỘ hoặc bác TOÀN BỘ, chứ không cắt đôi.
Vì vậy không được áp một mặc định duy nhất cho mọi yêu cầu; phải xét theo loại yêu cầu.'''

# Model chưa bao giờ được nhìn thấy schema: USE_STRUCTURED_OUTPUTS=False nên không có
# grammar ép, mà prompt cũ chỉ nói "Chỉ trả JSON theo schema" — hệ quả là model tự đặt tên
# khóa (reliefs/issue_map/claim_type/...) và validate_issue_map vứt sạch => mapper_fallback
# 50/50 ở mọi lần chạy. Khối dưới in ĐÚNG tên khóa + enum cho model.
ISSUE_MAPPER_SCHEMA_BLOCK = """ĐỊNH DẠNG BẮT BUỘC — in ĐÚNG một object JSON, dùng CHÍNH XÁC các tên khóa sau, không đổi tên, không thêm khóa nào khác:
{
  "claims": [
    {
      "claim_id": "C1",
      "request_quote": "<đoạn copy nguyên văn liên tục từ CASE_QUERY>",
      "request_type": "PRINCIPAL",
      "importance": "MATERIAL"
    }
  ]
}
Mỗi claim BẮT BUỘC đủ 4 khóa đúng tên: claim_id, request_quote, request_type, importance.
"request_type" là LOẠI yêu cầu (KHÔNG phải mức quan trọng), chỉ nhận đúng một trong:
- PRINCIPAL: tiền gốc, nợ gốc, tiền vay/hụi/phường/công nợ phải trả.
- INTEREST: lãi, lãi suất, lãi chậm trả.
- PENALTY: phạt vi phạm, phạt cọc, án phí, lệ phí.
- DAMAGES: bồi thường thiệt hại, chi phí điều trị/sửa chữa, tổn thất tinh thần, thu nhập bị mất.
- PROPERTY: đòi lại hoặc giao trả ĐÚNG một tài sản/thửa đất cụ thể, công nhận quyền sử dụng, xử lý/phát mại tài sản thế chấp, hủy giấy chứng nhận.
- DIVISION: yêu cầu CHIA được theo tỷ lệ hoặc theo phần — chia di sản/thừa kế, chia tài sản chung, xác định lại ranh giới hoặc diện tích.
- TRANSACTION: hủy, tuyên vô hiệu, công nhận, hoặc buộc tiếp tục thực hiện hợp đồng/giao dịch.
- OTHER: yêu cầu không thuộc các nhóm trên.
"importance" chỉ nhận MATERIAL hoặc SECONDARY.
Ví dụ output hợp lệ:
{"claims":[{"claim_id":"C1","request_quote":"buộc trả nợ gốc 500.000.000 đồng","request_type":"PRINCIPAL","importance":"MATERIAL"},{"claim_id":"C2","request_quote":"tiền lãi theo hợp đồng","request_type":"INTEREST","importance":"SECONDARY"}]}
Không in giải thích, không in reasoning, không in markdown, không in thẻ think."""

ISSUE_MAPPER_SYSTEM_PROMPT = f'''Bạn là Issue Mapper trung lập.
Chỉ inventory relief mà A thực sự yêu cầu trong CASE_QUERY. Không tạo legal issue, defense hay counterclaim thành claim.
Tách RIÊNG từng khoản có thể được toà quyết định độc lập: tiền gốc, lãi, phạt, TỪNG khoản thiệt hại/chi phí khác nhau, hủy/công nhận giao dịch, trả tài sản. Ví dụ "chi phí sửa xe" và "viện phí" là HAI claim tách biệt. Khi CASE_QUERY nối nhiều khoản bằng "và" hoặc dấu phẩy, hãy tách thành nhiều claim.
request_quote phải là đoạn liên tục copy từ CASE_QUERY (mỗi claim trích đúng phần của mình).
TRƯỚC KHI IN, rà lại CASE_QUERY một lượt từ đầu đến cuối: mỗi động từ yêu cầu (yêu cầu / buộc / đề nghị / xin / đòi) và mỗi khoản tiền riêng biệt được nhắc tới phải nằm trong đúng một claim. Nếu A đòi cả nợ gốc lẫn lãi, đó là HAI claim. Nếu A đòi một tài sản và kèm khoản tiền thay thế khi không giao được, đó là HAI claim.
Ngược lại, nếu vụ việc thật sự chỉ có một yêu cầu duy nhất thì trả đúng một claim — tuyệt đối không bịa thêm relief không có trong CASE_QUERY.
MATERIAL gồm tiền gốc, quyền/tài sản/giao dịch chính, trả tài sản, hủy hoặc công nhận giao dịch.
SECONDARY gồm lãi, phạt, án phí và chi phí phụ.
{GROUNDING_RULES}
{ISSUE_MAPPER_SCHEMA_BLOCK}'''

JUDGE_SYSTEM_PROMPT = f'''Bạn là Legal Judge dự đoán kết quả tranh chấp dân sự Việt Nam.
Đọc kỹ CASE_QUERY và luật; issue map chỉ là inventory hỗ trợ, có thể chưa hoàn hảo.
BẮT BUỘC cân nhắc hai chiều trước khi quyết mỗi claim: (1) căn cứ MẠNH NHẤT để A được chấp nhận; (2) căn cứ MẠNH NHẤT để toà GIẢM hoặc BÁC một phần (thiếu chứng từ chứng minh mức tiền, lỗi hỗn hợp, mức đòi cao hơn thiệt hại thực, nghĩa vụ đối ứng của A). Chỉ FULL_ACCEPT khi CẢ trách nhiệm LẪN mức yêu cầu đều gần như chắc chắn.
Mỗi claim dùng một outcome (phải trung thực vì nhãn cuối được suy ra từ chúng):
- FULL_ACCEPT: A gần như chắc chắn được chấp nhận TOÀN BỘ claim (cả trách nhiệm và mức tiền).
- PARTIAL_A_LEAN: A thắng phần lớn nhưng khả năng bị giảm/không toàn bộ (điển hình cho yêu cầu bồi thường bằng tiền còn tranh cãi về mức).
- PARTIAL_B_LEAN: claim bị chia nhưng B giữ phần lớn hơn.
- REJECT: A gần như bị bác toàn bộ claim.
Với MỖI claim phải cân nhắc ĐỦ BA khả năng (chấp nhận toàn bộ / chấp nhận một phần / bác) rồi mới chọn; không ép về nhị phân và cũng không mặc định PARTIAL.
Chọn outcome theo request_type của claim:
- DAMAGES: mức tiền do A tự ước tính. Trách nhiệm rõ nhưng mức còn tranh cãi -> PARTIAL_A_LEAN; không có căn cứ trách nhiệm -> REJECT.
- PRINCIPAL, INTEREST: số tiền tính được từ hợp đồng/giấy vay/sao kê. Nếu nghĩa vụ và cách tính có căn cứ -> FULL_ACCEPT (đừng hạ xuống PARTIAL chỉ vì con số lớn); nếu khoản nợ bị phản bác hoặc không chứng minh được -> REJECT; chỉ PARTIAL khi chỉ ra được phần nào được thừa nhận và phần nào bị bác.
- PROPERTY, TRANSACTION: relief nguyên khối (đòi lại đúng một thửa đất/tài sản, hủy hoặc công nhận một giao dịch cụ thể) thường KHÔNG cắt đôi được -> FULL_ACCEPT hoặc REJECT tuỳ bên nào có căn cứ pháp lý mạnh hơn (giấy chứng nhận, hợp đồng công chứng, thời hiệu, người thứ ba ngay tình, thực tế quản lý). Nhưng nếu toà có thể công nhận MỘT PHẦN diện tích/giá trị thì vẫn chọn PARTIAL.
- DIVISION: relief CHIA ĐƯỢC (chia di sản/thừa kế, chia tài sản chung, xác định lại ranh giới/diện tích). Ở nhóm này PARTIAL là kết quả BÌNH THƯỜNG NHẤT: toà hiếm khi chia đúng y nguyên tỷ lệ A đề nghị và cũng hiếm khi bác sạch. Chỉ REJECT khi A hoàn toàn không có quyền hưởng, chỉ FULL_ACCEPT khi phương án chia của A được giữ nguyên.
- PENALTY và các khoản phụ: toà hay điều chỉnh, PARTIAL là bình thường.
PARTIAL có HAI chiều, phải dùng cả hai: PARTIAL_A_LEAN khi phần A đạt được LỚN HƠN phần bị bác; PARTIAL_B_LEAN khi phần A đạt được NHỎ HƠN phần bị bác (ví dụ A đòi 10 phần chỉ được 3, hoặc A thắng về nguyên tắc nhưng mất phần lớn giá trị). Đừng mặc định mọi PARTIAL đều là PARTIAL_A_LEAN.
Chọn PARTIAL_A_LEAN hay PARTIAL_B_LEAN thì phải nêu được cụ thể phần nào được chấp nhận và phần nào bị bác; nếu không nêu được thì đó không phải PARTIAL.
Không kết luận FULL_ACCEPT chỉ vì thấy trách nhiệm của B mà chưa xét mức yêu cầu. Không kết luận REJECT chỉ vì bản tóm tắt thiếu tài liệu.
TRƯỚC KHI chốt prediction là A_WIN hoặc B_WIN, kiểm tra lại một lần: có khoản nào A đòi mà bị cắt, bị giảm hoặc bị bác không, và có khoản nào A vẫn giữ được không? Nếu có cả hai thì nhãn đúng là PARTIAL_A_WIN hoặc PARTIAL_B_WIN.
Ưu tiên điều luật khớp trực tiếp tình tiết; đánh giá theo GIÁ TRỊ phần MATERIAL, không đếm claim thô.
{LABEL_GUIDE}
{GROUNDING_RULES}
Trả JSON theo schema, chỉ in một object JSON với đúng các khóa: claim_outcomes (mảng, mỗi phần tử có claim_id, outcome, legal_basis_ranks), prediction (nhãn tổng thể nhất quán với các outcome), confidence (0..1).'''

DIRECT_SYSTEM_PROMPT = f'''Bạn là bộ phân loại dự phòng cho tranh chấp dân sự Việt Nam.
Đọc CASE_QUERY và {TOP_K} điều luật rồi chọn đúng một nhãn. Không dùng gold label hoặc dữ liệu đánh giá.
{LABEL_GUIDE}
{GROUNDING_RULES}
Chỉ trả compact JSON theo schema.'''


def build_law_context(laws):
    blocks = []
    for law in laws:
        content = law['content_Article'][:MAX_ARTICLE_CHARS]
        blocks.append(
            f"[{law['rank']}] law_id={law['law_id']} | Điều {law['article_no']} | aid={law['aid']}\n"
            f'{content}'
        )
    return '\n\n'.join(blocks)


def build_issue_prompt(case_query, laws):
    return f'''CASE_QUERY:
{case_query.strip()}

{TOP_K} ĐIỀU LUẬT:
{build_law_context(laws)}

Inventory các relief của A theo thứ tự xuất hiện.'''


def build_judge_prompt(case_query, laws, issue_map, case_facts=''):
    facts_block = ('\n\nTHÔNG TIN VỤ VIỆC (trích đoạn hồ sơ, evidence bổ sung):\n'
                   + case_facts.strip()) if case_facts else ''
    return f'''CASE_QUERY:
{case_query.strip()}{facts_block}

{TOP_K} ĐIỀU LUẬT:
{build_law_context(laws)}

ISSUE MAP:
{json.dumps(issue_map, ensure_ascii=False, separators=(',', ':'))}

Quyết định từng claim và prediction tổng thể. prediction phải phản ánh giá trị phần MATERIAL, không phải số claim.'''


def build_direct_prompt(case_query, laws, case_facts=''):
    facts_block = ('\n\nTHÔNG TIN VỤ VIỆC (trích đoạn hồ sơ, evidence bổ sung):\n'
                   + case_facts.strip()) if case_facts else ''
    return f'''CASE_QUERY:
{case_query.strip()}{facts_block}

{TOP_K} ĐIỀU LUẬT:
{build_law_context(laws)}

Chọn nhãn dự đoán cuối cùng.'''


def build_messages(system_prompt, user_prompt):
    # Ton trong khuyen nghi vendor: model dat use_system_prompt=False (vd DeepSeek-R1)
    # gop system vao user turn. Noi dung prompt GIU NGUYEN, chi doi cach dong goi message.
    if ACTIVE_PROFILE['use_system_prompt']:
        return [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ]
    return [{'role': 'user', 'content': system_prompt + '\n\n' + user_prompt}]


def count_message_tokens(messages):
    # Qwen: giu nguyen (enable_thinking=False nhu cu). Model khac: bo kwarg de tokenizer khong ken.
    _kw = {'enable_thinking': False} if 'qwen' in MODEL_ID.lower() else {}
    return len(prompt_tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        **_kw,
    ))


def _normalize_match(text):
    normalized = unicodedata.normalize('NFKC', str(text or '')).casefold()
    return re.sub(r'[\W_]+', ' ', normalized, flags=re.UNICODE).strip()


# Dong tu/danh tu chi YEU CAU. Da bo cac tu qua ngan de tranh dinh nham sau khi khu
# dau: 'doi' dung mot minh trung voi 'doi' trong 'doi voi', 'nhan' trung 'nhan dan'.
_RELIEF_MARKERS = (
    'yeu cau', 'buoc', 'de nghi', 'doi lai', 'doi boi thuong', 'chia', 'tra', 'tra lai',
    'tra no', 'hoan tra', 'thanh toan', 'boi thuong', 'cong nhan', 'huy', 'huy bo',
    'giao', 'giao tra', 'lai', 'an phi', 'tiep tuc thuc hien', 'cham dut', 'thu hoi',
    'den bu', 'xac dinh lai', 'cap duong',
)
# Cum mo ta DOI TUONG tranh chap, khong phai relief -> khong duoc thanh claim.
_TOPIC_MARKERS = ('tranh chap', 'vu an', 'quan he phap luat')


def _is_query_span(candidate, case_query):
    """Chan hallucinate, nhung khong phat model vi dien dat lai dong tu dan.

    Do tren raw output lan chay truoc: 6/115 claim bi vut, trong do 4 la relief CO
    THAT chi khac cach dan (model viet 'buoc bi don giao nha dat' trong khi query
    viet 'yeu cau bi don giao nha dat'), 2 la cau mo ta doi tuong tranh chap (dung
    ra phai vut). Nen: van uu tien trich nguyen van; neu khong, chi chap nhan khi
    >=80% tu cua quote co mat trong query VA quote thuc su chua tu chi yeu cau VA
    khong phai cum mo ta tranh chap. Da test tren raw output that: 4 relief that
    duoc cuu, 2 cum mo ta van bi loai, moi quote bia dat deu bi tu choi.
    """
    candidate_norm = _normalize_match(candidate)
    query_norm = _normalize_match(case_query)
    if not candidate_norm:
        return False
    if candidate_norm in query_norm:
        return True
    padded = ' ' + _strip_accents(candidate_norm) + ' '
    if any((' ' + marker + ' ') in padded for marker in _TOPIC_MARKERS):
        return False
    tokens = candidate_norm.split()
    if len(tokens) < 2:
        return False
    query_tokens = set(query_norm.split())
    coverage = sum(1 for token in tokens if token in query_tokens) / len(tokens)
    has_relief = any((' ' + marker + ' ') in padded for marker in _RELIEF_MARKERS)
    return coverage >= 0.8 and has_relief


def fallback_issue_map(case_query, reason):
    return {
        'claims': [{
            'claim_id': 'C1',
            'request_quote': case_query.strip(),
            'request_type': 'OTHER',
            'importance': 'MATERIAL',
        }],
        'mapper_fallback': True,
        'mapper_fallback_reason': str(reason),
    }


VALID_REQUEST_TYPES = ('PRINCIPAL', 'INTEREST', 'PENALTY', 'DAMAGES',
                       'PROPERTY', 'DIVISION', 'TRANSACTION', 'OTHER')
# FIX: INTEREST khong con tu dong SECONDARY. Trong tranh chap tin dung, lai la mot
# phan gia tri lon va viec toa cat lai chinh la thu bien vu an thanh PARTIAL_A_WIN;
# xep no la SECONDARY khien nhan bi keo len A_WIN (v11: case_6663, case_3241).
SECONDARY_BY_TYPE = {'PENALTY'}

# Tên khóa model hay dùng thay cho tên trong schema (đo trên raw output lần chạy trước).
_QUOTE_KEYS = ('request_quote', 'quote', 'relief_quote', 'claim_quote', 'request',
               'specific_relief', 'claim', 'issue')
_TYPE_KEYS = ('request_type', 'claim_type', 'type', 'claim_category', 'category',
              'issue_type', 'claim_subtype', 'sub_category', 'classification')
_IMPORTANCE_KEYS = ('importance', 'materiality', 'importance_level', 'material',
                    'material_claim', 'secondary', 'secondary_material')
_TYPE_ALIASES = {
    'PRINCIPAL': 'PRINCIPAL', 'DEBT': 'PRINCIPAL', 'LOAN': 'PRINCIPAL', 'MONEY': 'PRINCIPAL',
    'GOC': 'PRINCIPAL', 'NO_GOC': 'PRINCIPAL', 'PAYMENT': 'PRINCIPAL',
    'INTEREST': 'INTEREST', 'LAI': 'INTEREST', 'LAI_SUAT': 'INTEREST',
    'PENALTY': 'PENALTY', 'FINE': 'PENALTY', 'PHAT': 'PENALTY', 'FEE': 'PENALTY',
    'DAMAGES': 'DAMAGES', 'DAMAGE': 'DAMAGES', 'COMPENSATION': 'DAMAGES',
    'BOI_THUONG': 'DAMAGES', 'THIET_HAI': 'DAMAGES', 'COST': 'DAMAGES',
    'PROPERTY': 'PROPERTY', 'ASSET': 'PROPERTY', 'LAND': 'PROPERTY', 'REAL_ESTATE': 'PROPERTY',
    'OWNERSHIP': 'PROPERTY', 'TAI_SAN': 'PROPERTY', 'DAT': 'PROPERTY',
    'DIVISION': 'DIVISION', 'INHERITANCE': 'DIVISION', 'PARTITION': 'DIVISION',
    'CHIA': 'DIVISION', 'THUA_KE': 'DIVISION', 'DI_SAN': 'DIVISION', 'BOUNDARY': 'DIVISION',
    'TRANSACTION': 'TRANSACTION', 'CONTRACT': 'TRANSACTION', 'AGREEMENT': 'TRANSACTION',
    'NULLIFICATION': 'TRANSACTION', 'HOP_DONG': 'TRANSACTION', 'GIAO_DICH': 'TRANSACTION',
    'OTHER': 'OTHER', 'MISC': 'OTHER', 'KHAC': 'OTHER',
}
# Backstop khi model không cho enum dùng được: suy request_type từ chính đoạn trích.
_TYPE_PATTERNS = (
    ('INTEREST', r'(lai suat|tien lai|lai trong han|lai qua han|lai cham tra|lai phat sinh|goc va lai|va lai)'),
    ('PENALTY', r'(phat vi pham|tien phat|phat coc|phat hop dong|phat cham|an phi|le phi)'),
    ('DAMAGES', r'(boi thuong|thiet hai|chi phi dieu tri|chi phi chua|vien phi|ton that tinh than|chi phi sua|mai tang|cap duong|thu nhap bi mat|cong suc)'),
    ('TRANSACTION', r'(huy (bo )?(hop dong|giao dich|van ban|thoa thuan)|vo hieu|cong nhan (hop dong|giao dich|hieu luc)|tiep tuc thuc hien|cham dut hop dong|chuyen nhuong)'),
    ('DIVISION', r'(chia (deu |doi |theo )?(di san|thua ke|tai san|gia tri|phan)|phan chia|thua ke theo phap luat|di san thua ke|tai san chung|ranh gioi|dien tich thuc te)'),
    ('PROPERTY', r'(quyen su dung dat|su dung dat|tra (lai )?(nha|dat|tai san|phan dat|dien tich)|giao (tra|lai )?dat|thua ke|di san|chia (deu )?(tai san|di san|gia tri)|so do|giay chung nhan|thao do|di doi|lan chiem|so huu|phat mai|tai san the chap|xu ly tai san)'),
    ('PRINCIPAL', r'(no goc|von goc|tien vay|tien hui|tien phuong|cong no|tra no|thanh toan|hoan tra|lien doi tra|buoc tra|nghia vu tra)'),
)


def _strip_accents(text):
    decomposed = unicodedata.normalize('NFD', str(text or '').casefold())
    return ''.join(ch for ch in decomposed if unicodedata.category(ch) != 'Mn').replace('đ', 'd')


def _first_list_of_dicts(obj, depth=0):
    """Model hay đặt danh sách claim dưới tên khóa khác (reliefs/issue_map/...).
    Nhận danh sách dict dài nhất; nếu object chính là một claim đơn thì bọc lại."""
    if not isinstance(obj, dict) or depth > 2:
        return None
    lists = [v for v in obj.values()
             if isinstance(v, list) and v and all(isinstance(i, dict) for i in v)]
    if lists:
        return max(lists, key=len)
    if any(k in obj for k in _QUOTE_KEYS):
        return [obj]
    for value in obj.values():
        nested = _first_list_of_dicts(value, depth + 1)
        if nested:
            return nested
    return None


def _pick_field(raw, keys):
    for key in keys:
        if key in raw and raw[key] not in (None, ''):
            return raw[key]
    return None


def _infer_type_from_quote(quote):
    if re.search(r'lãi', str(quote or ''), flags=re.I):
        return 'INTEREST'
    flat = re.sub(r'[\W_]+', ' ', _strip_accents(quote)).strip()
    for name, pattern in _TYPE_PATTERNS:
        if re.search(pattern, flat):
            return name
    return 'OTHER'


# Relief CHIA DUOC bi model gan nham sang PROPERTY/TRANSACTION thi se an chinh sach
# all-or-nothing va sup ve A_WIN/B_WIN (v11: case_614, case_7467). Dau hieu "chia di san /
# chia thua ke / chia tai san chung / ranh gioi" la xac dinh du de ghi de.
_DIVISION_OVERRIDE = re.compile(
    r'(chia (deu |doi |theo )?(di san|thua ke|tai san|gia tri|phan)|phan chia'
    r'|thua ke theo phap luat|di san thua ke|tai san chung|ranh gioi|dien tich thuc te)'
)


def _pick_request_type(raw, quote):
    if _DIVISION_OVERRIDE.search(re.sub(r'[\W_]+', ' ', _strip_accents(quote)).strip()):
        return 'DIVISION'
    value = _pick_field(raw, _TYPE_KEYS)
    token = re.sub(r'[\W]+', '_', str(value or '').strip().upper()).strip('_')
    # Model rất hay nhét MATERIAL/SECONDARY vào ô type -> đó là importance, không phải type.
    if token and token not in {'MATERIAL', 'SECONDARY', 'PRIMARY'}:
        if token in VALID_REQUEST_TYPES:
            return token
        if token in _TYPE_ALIASES:
            return _TYPE_ALIASES[token]
        for alias, canonical in _TYPE_ALIASES.items():
            if alias in token:
                return canonical
    return _infer_type_from_quote(quote)


def _pick_importance(raw, request_type):
    if request_type in SECONDARY_BY_TYPE:
        return 'SECONDARY'
    # Model rat hay dao cho: nhet MATERIAL/SECONDARY vao o type. Van la tin hieu importance.
    for key in _TYPE_KEYS:
        token = str(raw.get(key, '') or '').strip().upper()
        if token in {'MATERIAL', 'PRIMARY'}:
            return 'MATERIAL'
        if token == 'SECONDARY':
            return 'SECONDARY'
    for key in _IMPORTANCE_KEYS:
        if key not in raw or raw[key] in (None, ''):
            continue
        value = raw[key]
        if isinstance(value, bool):
            if key == 'secondary':
                return 'SECONDARY' if value else 'MATERIAL'
            return 'MATERIAL' if value else 'SECONDARY'
        token = str(value).strip().upper()
        if 'SECOND' in token or token in {'PHU', 'FALSE', 'NO', 'LOW'}:
            return 'SECONDARY'
        if 'MATERIAL' in token or 'CHINH' in token or token in {'TRUE', 'YES', 'MAIN', 'PRIMARY', 'HIGH'}:
            return 'MATERIAL'
    return 'MATERIAL'


def validate_issue_map(obj, case_query):
    """Chấp nhận tên khóa lệch schema. Grounding vẫn bắt buộc qua _is_query_span:
    ưu tiên trích nguyên văn, chỉ nới đúng mức cho trường hợp model đổi động từ dẫn
    ('buộc' thay cho 'yêu cầu'). Quote bịa đặt vẫn bị từ chối — đây vẫn là chốt
    chống hallucinate, không được nới thêm."""
    raw_claims = obj.get('claims')
    if not isinstance(raw_claims, list) or not raw_claims:
        raw_claims = _first_list_of_dicts(obj)
    if not isinstance(raw_claims, list):
        raise ValueError('claims must be a list')
    claims, seen = [], set()
    for raw in raw_claims:
        if not isinstance(raw, dict):
            continue
        quote = str(_pick_field(raw, _QUOTE_KEYS) or '').strip()
        if not _is_query_span(quote, case_query):
            continue
        request_type = _pick_request_type(raw, quote)
        importance = _pick_importance(raw, request_type)
        key = (_normalize_match(quote), request_type)
        if key in seen:
            continue
        seen.add(key)
        claims.append({
            'claim_id': f'C{len(claims) + 1}',
            'request_quote': quote,
            'request_type': request_type,
            'importance': importance,
        })
        if len(claims) >= 8:  # khớp maxItems của ISSUE_SCHEMA
            break
    if not claims:
        raise ValueError('no grounded unique claims survived validation')
    if not any(claim['importance'] == 'MATERIAL' for claim in claims):
        claims[0]['importance'] = 'MATERIAL'
    return {'claims': claims, 'mapper_fallback': False, 'mapper_fallback_reason': None}


def _json_object(text):
    cleaned = (text or '')
    # Thinking mode phát ra <think>...</think> trước JSON; bỏ đi như base.
    cleaned = re.sub(r'<think>.*?</think>', '', cleaned, flags=re.I | re.S)
    if '</think>' in cleaned:
        cleaned = cleaned.rsplit('</think>', 1)[-1]
    cleaned = re.sub(r'^```(?:json)?\s*|\s*```$', '', cleaned.strip(), flags=re.I | re.S).strip()
    try:
        obj = json.loads(cleaned)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    decoder = json.JSONDecoder()
    for match in re.finditer(r'\{', cleaned):
        try:
            obj, _ = decoder.raw_decode(cleaned[match.start():])
        except Exception:
            continue
        if isinstance(obj, dict):
            return obj
    raise ValueError('no complete JSON object')


def derive_label(issue_map, claim_outcomes):
    '''Suy nhãn 4 lớp từ outcome của từng claim, trọng số theo importance.
    PADO đúng nghĩa: nhãn tổng thể là hệ quả của các quyết định claim, không phải
    nhãn model tự khai. Đây là điều làm PARTIAL xuất hiện đúng cấu trúc.'''
    weight_by_id = {
        c['claim_id']: (2.0 if c.get('importance') == 'MATERIAL' else 1.0)
        for c in issue_map['claims']
    }
    a_share = {'FULL_ACCEPT': 1.0, 'PARTIAL_A_LEAN': 0.65, 'PARTIAL_B_LEAN': 0.35, 'REJECT': 0.0}
    num = den = 0.0
    for co in claim_outcomes:
        w = weight_by_id.get(co['claim_id'], 1.0)
        num += w * a_share.get(co['outcome'], 0.5)
        den += w
    if den == 0:
        return None
    s = num / den
    if s >= 0.85:
        return 'A_WIN'
    if s >= 0.55:
        return 'PARTIAL_A_WIN'
    if s >= 0.30:
        return 'PARTIAL_B_WIN'
    return 'B_WIN'


def consensus_claim_outcomes(votes, issue_map):
    """Tang 1 (majority-at-claim): gop outcome cua TUNG claim qua cac phieu roi moi
    derive, thay vi derive tren mot phieu dai dien duy nhat.

    Ly do: tren run Qwen3.5-9B v15 chi 53% claim duoc ca 3 phieu cho cung outcome; chon
    1 phieu dai dien khuech dai nhieu sampling, con gop o muc claim dan nhieu o don vi
    nho nhat. Do doi chung (cung outcomes): phieu dai dien strict .580/F1 .547 ->
    majority-at-claim .640/.622 (paired bootstrap P(tot hon) = .939). Thuan hau ky.

    Tie-break khi cac phieu bat dong hoan toan: trong cac outcome thang phieu, chon
    outcome ON HOA nhat (a_share gan 0.65) - xac dinh, khong phu thuoc thu tu phieu.
    """
    a_share = {'FULL_ACCEPT': 1.0, 'PARTIAL_A_LEAN': 0.65, 'PARTIAL_B_LEAN': 0.35, 'REJECT': 0.0}
    order, tally = [], {}
    for vote in votes:
        for item in vote.get('claim_outcomes', []) or []:
            claim_id = item.get('claim_id')
            outcome = item.get('outcome')
            if claim_id is None or outcome is None:
                continue
            if claim_id not in tally:
                tally[claim_id] = {}
                order.append(claim_id)
            tally[claim_id][outcome] = tally[claim_id].get(outcome, 0) + 1
    consensus = []
    for claim_id in order:
        counts = tally[claim_id]
        top_n = max(counts.values())
        winners = [oc for oc in counts if counts[oc] == top_n]
        pick = min(winners, key=lambda oc: abs(a_share.get(oc, 0.5) - 0.65))
        consensus.append({'claim_id': claim_id, 'outcome': pick})
    return consensus


def validate_judge(obj, issue_map, retrieved_laws):
    prediction = str(obj.get('prediction', '')).strip().upper()
    if prediction not in LABELS:
        raise ValueError(f'invalid prediction {prediction!r}')
    try:
        confidence = min(1.0, max(0.0, float(obj.get('confidence', 0.0))))
    except Exception:
        confidence = 0.0

    valid_outcomes = {'FULL_ACCEPT', 'PARTIAL_A_LEAN', 'PARTIAL_B_LEAN', 'REJECT'}
    expected_ids = [claim['claim_id'] for claim in issue_map['claims']]
    raw_by_id = {}
    for item in obj.get('claim_outcomes', []) if isinstance(obj.get('claim_outcomes'), list) else []:
        if not isinstance(item, dict):
            continue
        claim_id = str(item.get('claim_id', '')).strip()
        outcome = str(item.get('outcome', '')).strip().upper()
        if claim_id not in expected_ids or outcome not in valid_outcomes or claim_id in raw_by_id:
            continue
        ranks = []
        for rank in item.get('legal_basis_ranks', []) if isinstance(item.get('legal_basis_ranks'), list) else []:
            try:
                rank = int(rank)
            except Exception:
                continue
            if 1 <= rank <= len(retrieved_laws) and rank not in ranks:
                ranks.append(rank)
        raw_by_id[claim_id] = {
            'claim_id': claim_id,
            'outcome': outcome,
            'legal_basis_ranks': ranks[:3],
        }

    default_outcome = {
        'A_WIN': 'FULL_ACCEPT',
        'PARTIAL_A_WIN': 'PARTIAL_A_LEAN',
        'PARTIAL_B_WIN': 'PARTIAL_B_LEAN',
        'B_WIN': 'REJECT',
    }[prediction]
    outcomes = [
        raw_by_id.get(claim_id, {
            'claim_id': claim_id,
            'outcome': default_outcome,
            'legal_basis_ranks': [],
        })
        for claim_id in expected_ids
    ]
    ranks = []
    for item in outcomes:
        for rank in item['legal_basis_ranks']:
            if rank not in ranks:
                ranks.append(rank)
    applied_laws = [
        {
            'law_id': str(retrieved_laws[rank - 1]['law_id']),
            'aid': int(retrieved_laws[rank - 1]['aid']),
            'reason': f'Legal Judge selected retrieved law rank {rank}',
        }
        for rank in ranks
    ]
    derived = derive_label(issue_map, outcomes)
    # 'prediction' o day van la nhan holistic cua tung phieu - aggregate_votes dung no
    # de chon phieu dai dien. Nhan CUOI CUNG cua case duoc suy tu claim_outcomes o cell
    # ket qua (xem 'decider' trong result). Comment cu khuyen khong dung derived la viet
    # cho thoi mapper hong (1 claim/case, derive suy bien) - dieu kien do khong con dung.
    return {
        'prediction': prediction,
        'derived_label': derived,
        'confidence': confidence,
        'claim_outcomes': outcomes,
        'applied_laws': applied_laws,
    }


def validate_direct(obj):
    prediction = str(obj.get('prediction', '')).strip().upper()
    if prediction not in LABELS:
        raise ValueError(f'invalid direct prediction {prediction!r}')
    try:
        confidence = min(1.0, max(0.0, float(obj.get('confidence', 0.0))))
    except Exception:
        confidence = 0.0
    return {'prediction': prediction, 'confidence': confidence}


def aggregate_votes(votes):
    if not votes:
        raise ValueError('cannot aggregate zero votes')
    counts = {label: 0 for label in LABELS}
    confidence_sum = {label: 0.0 for label in LABELS}
    for vote in votes:
        label = vote['prediction']
        counts[label] += 1
        confidence_sum[label] += vote['confidence']
    max_count = max(counts.values())
    candidates = [label for label in LABELS if counts[label] == max_count]
    # Hoa phieu = cac vote khong dong y = bat dinh. Truoc day tie-break la
    # -LABELS.index() nen luon roi ve A_WIN (LABELS[0]) - mot thien lech ve nhan
    # cuc doan dung luc bang chung yeu nhat. Gio hoa phieu thi nghieng ve nhan
    # trung dung (PARTIAL) truoc.
    tie_break_rank = {'PARTIAL_A_WIN': 3, 'PARTIAL_B_WIN': 2, 'A_WIN': 1, 'B_WIN': 0}
    candidates.sort(
        key=lambda label: (
            confidence_sum[label] / max(1, counts[label]),
            tie_break_rank[label],
        ),
        reverse=True,
    )
    prediction = candidates[0]
    selected = [vote for vote in votes if vote['prediction'] == prediction]
    representative = max(selected, key=lambda vote: vote['confidence'])
    return {
        'prediction': prediction,
        'confidence': sum(vote['confidence'] for vote in selected) / len(selected),
        'vote_counts': counts,
        'n_valid_votes': len(votes),
        'claim_outcomes': representative['claim_outcomes'],
        'applied_laws': representative['applied_laws'],
    }


def deterministic_stage_seed(eval_seed, case_id, stage):
    raw = f'{eval_seed}|{case_id}|{stage}'.encode('utf-8')
    return int.from_bytes(hashlib.sha256(raw).digest()[:4], 'big')


ISSUE_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'claims': {
            'type': 'array',
            'minItems': 1,
            'maxItems': 8,
            'items': {
                'type': 'object',
                'additionalProperties': False,
                'properties': {
                    'claim_id': {'type': 'string', 'pattern': '^C[1-8]$'},
                    'request_quote': {'type': 'string', 'minLength': 1, 'maxLength': 600},
                    'request_type': {
                        'type': 'string',
                        'enum': ['PRINCIPAL', 'INTEREST', 'PENALTY', 'DAMAGES', 'PROPERTY', 'DIVISION', 'TRANSACTION', 'OTHER'],
                    },
                    'importance': {'type': 'string', 'enum': ['MATERIAL', 'SECONDARY']},
                },
                'required': ['claim_id', 'request_quote', 'request_type', 'importance'],
            },
        },
    },
    'required': ['claims'],
}

CLAIM_OUTCOME_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'claim_id': {'type': 'string', 'pattern': '^C[1-8]$'},
        'outcome': {
            'type': 'string',
            'enum': ['FULL_ACCEPT', 'PARTIAL_A_LEAN', 'PARTIAL_B_LEAN', 'REJECT'],
        },
        'legal_basis_ranks': {
            'type': 'array',
            'minItems': 1,
            'maxItems': 3,
            'items': {'type': 'integer', 'minimum': 1, 'maximum': 10},
        },
    },
    'required': ['claim_id', 'outcome', 'legal_basis_ranks'],
}

JUDGE_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'claim_outcomes': {'type': 'array', 'minItems': 1, 'maxItems': 8, 'items': CLAIM_OUTCOME_SCHEMA},
        'prediction': {'type': 'string', 'enum': LABELS},
        'confidence': {'type': 'number', 'minimum': 0.0, 'maximum': 1.0},
    },
    'required': ['claim_outcomes', 'prediction', 'confidence'],
}

DIRECT_SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'prediction': {'type': 'string', 'enum': LABELS},
        'confidence': {'type': 'number', 'minimum': 0.0, 'maximum': 1.0},
    },
    'required': ['prediction', 'confidence'],
}

HARD_FALLBACK_LABEL = os.getenv('ALQAC_HARD_FALLBACK_LABEL', 'PARTIAL_A_WIN').strip().upper()
if HARD_FALLBACK_LABEL not in LABELS:
    raise ValueError(f'Invalid ALQAC_HARD_FALLBACK_LABEL={HARD_FALLBACK_LABEL!r}')

PIPELINE_CONTRACT = {
    'pipeline_version': PADO_PIPELINE_VERSION,
    'stages': ['issue_mapper', 'legal_judge_3vote', 'direct_fallback'],
    'issue_schema': ISSUE_SCHEMA,
    'judge_schema': JUDGE_SCHEMA,
    'direct_schema': DIRECT_SCHEMA,
    'judge_votes': JUDGE_VOTES,
    'disable_any_whitespace': USE_STRUCTURED_OUTPUTS,
    'hard_fallback_label': HARD_FALLBACK_LABEL,
}
PIPELINE_CONTRACT_HASH = hashlib.sha256(
    json.dumps(PIPELINE_CONTRACT, ensure_ascii=False, sort_keys=True).encode('utf-8')
).hexdigest()


def aggregate_stage_usage(stage_usages):
    values = list(stage_usages.values())
    return {
        'input_tokens': sum(item['input_tokens'] for item in values),
        'output_tokens': sum(item['output_tokens'] for item in values),
        'total_tokens': sum(item['total_tokens'] for item in values),
        'hit_max_new_tokens': any(item.get('hit_max_new_tokens', False) for item in values),
        'stage_count': len(values),
        'stages': stage_usages,
    }


def vllm_generate(messages_list, seeds, schema, max_tokens, enable_thinking=None):
    if len(messages_list) != len(seeds):
        raise ValueError('messages_list and seeds length mismatch')
    outputs = _ensure_remote().generate_batch.remote(
        messages_list, seeds, schema, max_tokens, enable_thinking
    )
    return [(text, usage) for text, usage in outputs]


BENCHMARK_MANIFEST = {
    'benchmark_version': BENCHMARK_VERSION,
    'benchmark_protocol': BENCHMARK_PROTOCOL,
    'pipeline_version': PADO_PIPELINE_VERSION,
    'pipeline_contract_hash': PIPELINE_CONTRACT_HASH,
    'pipeline_stages': PIPELINE_CONTRACT['stages'],
    'model_name': MODEL_NAME,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'generation': RESOLVED_GENERATION_CONFIG,
    'model_context_limit': MODEL_CONTEXT_LIMIT,
    'dtype': MODEL_DTYPE,
    'full_gpu_no_quantization': True,
    'dataset_path': str(DATA_PATH),
    'dataset_sha256': hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    'num_cases': len(inference_cases),
    'labels': LABELS,
    'retrieval_source': RETRIEVAL_SOURCE,
    'retrieval_path': str(RETRIEVAL_PATH),
    'retrieval_sha256': hashlib.sha256(RETRIEVAL_PATH.read_bytes()).hexdigest(),
    'top_k_laws': TOP_K,
    'retrieval_signature': retrieval_signature,
    'max_input_tokens': MAX_INPUT_TOKENS,
    'max_article_chars': MAX_ARTICLE_CHARS,
    'eval_seeds': EVAL_SEEDS,
    'coverage_policy': 'mapper_fallback + direct_fallback + explicit hard fallback always emit a label',
    'prompt_hashes': {
        'issue_mapper': hashlib.sha256(ISSUE_MAPPER_SYSTEM_PROMPT.encode('utf-8')).hexdigest(),
        'legal_judge': hashlib.sha256(JUDGE_SYSTEM_PROMPT.encode('utf-8')).hexdigest(),
        'direct_fallback': hashlib.sha256(DIRECT_SYSTEM_PROMPT.encode('utf-8')).hexdigest(),
    },
}

# ---- Preflight ngan sach token (chay local, truoc khi dot GPU) ----
# Top-14 dai hon top-10 dang ke; kiem tra truoc de khong chet giua chung o checked_messages.
def _worst_issue_map(case_query):
    # Upper bound that: schema cho toi da 8 claim, va validate_issue_map bat moi request_quote
    # phai la doan trich tu case_query -> khong claim nao dai hon ca case_query.
    return {
        'claims': [{'claim_id': f'C{i}', 'request_quote': case_query.strip(),
                    'request_type': 'DAMAGES', 'importance': 'MATERIAL'} for i in range(1, 9)],
        'mapper_fallback': False,
    }


_budget = []
for _case in inference_cases:
    _msgs = build_messages(
        JUDGE_SYSTEM_PROMPT,
        build_judge_prompt(_case['case_query'], retrieval_cache[_case['case_id']],
                           _worst_issue_map(_case['case_query']), _case['case_facts']),
    )
    _budget.append((count_message_tokens(_msgs), _case['case_id']))
_max_in, _worst_case = max(_budget)
_headroom = MODEL_CONTEXT_LIMIT - MAX_NEW_TOKENS_JUDGE - _max_in
print(f'Preflight judge prompt (worst-case issue map): input TB='
      f'{sum(t for t, _ in _budget) // len(_budget)} | MAX={_max_in} ({_worst_case})'
      f' | con du {_headroom} token so voi {MODEL_CONTEXT_LIMIT} - {MAX_NEW_TOKENS_JUDGE}')
if _max_in > MAX_INPUT_TOKENS or _headroom < 0:
    raise RuntimeError(
        f'Ngan sach prompt khong du: input toi da {_max_in} token, gioi han '
        f'{MAX_INPUT_TOKENS}/{MODEL_CONTEXT_LIMIT} voi max_new={MAX_NEW_TOKENS_JUDGE}. '
        'Ha ALQAC_MAX_ARTICLE_CHARS (vd 3000) hoac ALQAC_MAX_EVIDENCE_CHARS roi chay lai cell 2-5.'
    )

print(
    'v12 compact ensemble ready | GPU:', GPU_TYPE,
    '| whitespace disabled: True',
    '| thinking:', ENABLE_THINKING,
    '| sampling:', SAMPLING,
    '| votes:', JUDGE_VOTES,
    '| contract:', PIPELINE_CONTRACT_HASH[:12],
    '| retrieval:', RETRIEVAL_SOURCE,
)


config.json:   0%|          | 0.00/3.13k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

Preflight judge prompt (worst-case issue map): input TB=2 | MAX=2 (case_9643) | con du 24574 token so voi 40960 - 16384
v12 compact ensemble ready | GPU: A100-40GB | whitespace disabled: True | thinking: True | sampling: {'temperature': 1.0, 'top_p': 0.95, 'top_k': 20, 'min_p': 0.0, 'repetition_penalty': 1.0} | votes: 3 | contract: 8e6b858d3266 | retrieval: file:retrieval_top14_private.json


In [6]:
_cid = sample_case['case_id']
_query = sample_case['case_query']
_facts = evidence_by_case.get(_cid, '')
_seed = EVAL_SEEDS[0]
_laws = retrieval_cache[_cid]
try:
    mapper_raw, mapper_usage = vllm_generate(
        [build_messages(ISSUE_MAPPER_SYSTEM_PROMPT, build_issue_prompt(_query, _laws))],
        [deterministic_stage_seed(_seed, _cid, 'issue_mapper')],
        ISSUE_SCHEMA,
        MAX_NEW_TOKENS_ISSUE,
        enable_thinking=False,  # khớp stage mapper của lần chạy thật
    )[0]
    try:
        issue_map = validate_issue_map(_json_object(mapper_raw), _query)
    except Exception as mapper_exc:
        issue_map = fallback_issue_map(_query, mapper_exc)

    vote_raw, vote_usage = vllm_generate(
        [build_messages(JUDGE_SYSTEM_PROMPT, build_judge_prompt(_query, _laws, issue_map, _facts))],
        [deterministic_stage_seed(_seed, _cid, 'legal_judge_vote_0')],
        JUDGE_SCHEMA,
        MAX_NEW_TOKENS_JUDGE,
    )[0]
    vote = validate_judge(_json_object(vote_raw), issue_map, _laws)
    print(
        'SMOKE OK | mapper_fallback:', issue_map['mapper_fallback'],
        '| claims:', len(issue_map['claims']),
        '| types:', [c['request_type'] for c in issue_map['claims']],
        '| prediction:', vote['prediction'],
        '| usage:', aggregate_stage_usage({'issue_mapper': mapper_usage, 'legal_judge_vote_0': vote_usage}),
    )
except Exception as exc:
    print('SMOKE WARNING:', repr(exc))


✓ Initialized. View run at https://modal.com/apps/dangkhoaichau/main/ap-8u5Ro5jtmrtw1fb8rE9QpL
⠋ Initializing...
⠦ Creating objects...
⠧ Creating objects...
└── 🔨 Created function RemoteLLM.*.
✓ Created objects.
└── 🔨 Created function RemoteLLM.*.
⠴ Loading images (1 containers initializing)... View app at 
⠇ Loading images (1 containers initializing)... View app at 
⠙ Loading images (1 containers initializing)... View app at 
⠼ Loading images (1 containers initializing)... View app at 
⠧ Loading images (1 containers initializing)... View app at 
⠋ Loading images (1 containers initializing)... View app at 
⠼ Loading images (1 containers initializing)... View app at 
⠧ Loading images (1 containers initializing)... View app at 
⠋ Loading images (1 containers initializing)... View app at 
⠸ Loading images (1 containers initializing)... View app at 
⠦ Loading images (1 containers initializing)... View app at 
⠏ Loading images (1 containers initializing)... View app at 
⠹ Loading images (1 

In [7]:
def slugify(text):
    return re.sub(r'[^a-z0-9]+', '-', text.lower()).strip('-')


def load_json(path, default):
    if not path.exists():
        return default
    try:
        return json.loads(path.read_text(encoding='utf-8'))
    except Exception:
        return default


def atomic_write_json(path, obj):
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding='utf-8')
    temp.replace(path)


def make_cache_key(model, case, laws, eval_seed):
    material = {
        'benchmark_version': BENCHMARK_VERSION,
        'benchmark_protocol': BENCHMARK_PROTOCOL,
        'pipeline_version': PADO_PIPELINE_VERSION,
        'pipeline_contract_hash': PIPELINE_CONTRACT_HASH,
        'model': model,
        'model_id': MODEL_ID,
        'model_revision': MODEL_REVISION,
        'generation': RESOLVED_GENERATION_CONFIG,
        'eval_seed': eval_seed,
        'case_id': case['case_id'],
        'case_query': case['case_query'],
        'case_facts': case.get('case_facts', ''),
        'laws': laws,
        'prompts': {
            'issue_mapper': ISSUE_MAPPER_SYSTEM_PROMPT,
            'legal_judge': JUDGE_SYSTEM_PROMPT,
            'direct_fallback': DIRECT_SYSTEM_PROMPT,
        },
        'schemas': {'issue': ISSUE_SCHEMA, 'judge': JUDGE_SCHEMA, 'direct': DIRECT_SCHEMA},
    }
    return hashlib.sha256(
        json.dumps(material, ensure_ascii=False, sort_keys=True).encode('utf-8')
    ).hexdigest()


manifest_path = OUTPUT_DIR / f'benchmark_manifest_{slugify(MODEL_NAME)}.json'
atomic_write_json(manifest_path, BENCHMARK_MANIFEST)
print('Benchmark manifest:', manifest_path)

MODEL = MODELS_TO_RUN[0]
EVAL_SEED = EVAL_SEEDS[0]
result_path = OUTPUT_DIR / f'predictions_{slugify(MODEL)}_seed-{EVAL_SEED}.json'
saved = load_json(result_path, {})

cache_keys, cases_run = {}, []
for case in inference_cases:
    cid = case['case_id']
    cache_key = make_cache_key(MODEL, case, retrieval_cache[cid], EVAL_SEED)
    cache_keys[cid] = cache_key
    old = saved.get(cid)
    if old and old.get('cache_key') == cache_key and old.get('prediction') in LABELS:
        continue
    cases_run.append(case)
print(f'=== v12 compact ensemble | seed={EVAL_SEED} | run {len(cases_run)}/{len(inference_cases)} cases ===')

state = {}
for case in cases_run:
    state[case['case_id']] = {
        'laws': retrieval_cache[case['case_id']],
        'raw': {},
        'usage': {},
        'issue_map': None,
        'votes': [],
        'stage_errors': [],
    }


def checked_messages(system_prompt, user_prompt, max_tokens):
    messages = build_messages(system_prompt, user_prompt)
    input_tokens = count_message_tokens(messages)
    if input_tokens > MAX_INPUT_TOKENS or input_tokens + max_tokens > MODEL_CONTEXT_LIMIT:
        raise ValueError(
            f'prompt budget exceeded: input={input_tokens}, output={max_tokens}, '
            f'limits={MAX_INPUT_TOKENS}/{MODEL_CONTEXT_LIMIT}'
        )
    return messages


started_all = time.time()

# Stage 1: mapper. Every failure is converted to a one-claim grounded fallback.
mapper_messages, mapper_seeds, mapper_cases = [], [], []
for case in cases_run:
    current = state[case['case_id']]
    try:
        mapper_messages.append(checked_messages(
            ISSUE_MAPPER_SYSTEM_PROMPT,
            build_issue_prompt(case['case_query'], current['laws']),
            MAX_NEW_TOKENS_ISSUE,
        ))
        mapper_seeds.append(deterministic_stage_seed(EVAL_SEED, case['case_id'], 'issue_mapper'))
        mapper_cases.append(case)
    except Exception as exc:
        current['stage_errors'].append({'stage': 'issue_mapper_input', 'error': repr(exc)})
        current['issue_map'] = fallback_issue_map(case['case_query'], exc)

if mapper_cases:
    try:
        mapper_outputs = vllm_generate(
            mapper_messages, mapper_seeds, ISSUE_SCHEMA, MAX_NEW_TOKENS_ISSUE,
            enable_thinking=False,
        )
        if len(mapper_outputs) != len(mapper_cases):
            raise RuntimeError(f'Mapper returned {len(mapper_outputs)}/{len(mapper_cases)} outputs')
        for case, (text, usage) in zip(mapper_cases, mapper_outputs):
            current = state[case['case_id']]
            current['raw']['issue_mapper'] = text
            current['usage']['issue_mapper'] = usage
            try:
                current['issue_map'] = validate_issue_map(_json_object(text), case['case_query'])
            except Exception as exc:
                current['stage_errors'].append({'stage': 'issue_mapper_output', 'error': repr(exc)})
                current['issue_map'] = fallback_issue_map(case['case_query'], exc)
    except Exception as exc:
        for case in mapper_cases:
            current = state[case['case_id']]
            current['stage_errors'].append({'stage': 'issue_mapper_infrastructure', 'error': repr(exc)})
            current['issue_map'] = fallback_issue_map(case['case_query'], exc)

for case in cases_run:
    current = state[case['case_id']]
    if current['issue_map'] is None:
        current['issue_map'] = fallback_issue_map(case['case_query'], 'mapper produced no state')

# Stage 2: three independent compact Judge votes. A failed vote does not kill the case.
for vote_index in range(JUDGE_VOTES):
    vote_messages, vote_seeds, vote_cases = [], [], []
    for case in cases_run:
        current = state[case['case_id']]
        try:
            vote_messages.append(checked_messages(
                JUDGE_SYSTEM_PROMPT,
                build_judge_prompt(
                    case['case_query'], current['laws'], current['issue_map'], case['case_facts']
                ),
                MAX_NEW_TOKENS_JUDGE,
            ))
            vote_seeds.append(deterministic_stage_seed(
                EVAL_SEED, case['case_id'], f'legal_judge_vote_{vote_index}'
            ))
            vote_cases.append(case)
        except Exception as exc:
            current['stage_errors'].append({
                'stage': f'legal_judge_vote_{vote_index}_input', 'error': repr(exc)
            })
    if not vote_cases:
        continue
    try:
        vote_outputs = vllm_generate(
            vote_messages, vote_seeds, JUDGE_SCHEMA, MAX_NEW_TOKENS_JUDGE
        )
        if len(vote_outputs) != len(vote_cases):
            raise RuntimeError(f'Judge vote {vote_index} returned {len(vote_outputs)}/{len(vote_cases)} outputs')
        for case, (text, usage) in zip(vote_cases, vote_outputs):
            current = state[case['case_id']]
            stage = f'legal_judge_vote_{vote_index}'
            current['raw'][stage] = text
            current['usage'][stage] = usage
            try:
                vote = validate_judge(
                    _json_object(text), current['issue_map'], current['laws']
                )
                vote['vote_index'] = vote_index
                current['votes'].append(vote)
            except Exception as exc:
                current['stage_errors'].append({'stage': stage, 'error': repr(exc)})
    except Exception as exc:
        for case in vote_cases:
            state[case['case_id']]['stage_errors'].append({
                'stage': f'legal_judge_vote_{vote_index}_infrastructure', 'error': repr(exc)
            })

# Direct fallback only for cases with zero usable Judge votes.
direct_cases = [case for case in cases_run if not state[case['case_id']]['votes']]
if direct_cases:
    direct_messages, direct_seeds, runnable_direct = [], [], []
    for case in direct_cases:
        current = state[case['case_id']]
        try:
            direct_messages.append(checked_messages(
                DIRECT_SYSTEM_PROMPT,
                build_direct_prompt(case['case_query'], current['laws'], case['case_facts']),
                MAX_NEW_TOKENS_DIRECT,
            ))
            direct_seeds.append(deterministic_stage_seed(EVAL_SEED, case['case_id'], 'direct_fallback'))
            runnable_direct.append(case)
        except Exception as exc:
            current['stage_errors'].append({'stage': 'direct_fallback_input', 'error': repr(exc)})
    if runnable_direct:
        try:
            direct_outputs = vllm_generate(
                direct_messages, direct_seeds, DIRECT_SCHEMA, MAX_NEW_TOKENS_DIRECT,
                enable_thinking=False,
            )
            if len(direct_outputs) != len(runnable_direct):
                raise RuntimeError(f'Direct fallback returned {len(direct_outputs)}/{len(runnable_direct)} outputs')
            for case, (text, usage) in zip(runnable_direct, direct_outputs):
                current = state[case['case_id']]
                current['raw']['direct_fallback'] = text
                current['usage']['direct_fallback'] = usage
                try:
                    current['direct'] = validate_direct(_json_object(text))
                except Exception as exc:
                    current['stage_errors'].append({'stage': 'direct_fallback', 'error': repr(exc)})
        except Exception as exc:
            for case in runnable_direct:
                state[case['case_id']]['stage_errors'].append({
                    'stage': 'direct_fallback_infrastructure', 'error': repr(exc)
                })

duration_all = round(time.time() - started_all, 3)
per_case_duration = round(duration_all / max(1, len(cases_run)), 3)
for case in cases_run:
    cid = case['case_id']
    current = state[cid]
    if current['votes']:
        verdict = aggregate_votes(current['votes'])
        prediction_source = (
            'judge_ensemble' if len(current['votes']) >= 2 else 'judge_single_vote'
        )
    elif current.get('direct'):
        verdict = {
            **current['direct'],
            'vote_counts': {label: 0 for label in LABELS},
            'n_valid_votes': 0,
            'claim_outcomes': [],
            'applied_laws': [],
        }
        prediction_source = 'direct_fallback'
    else:
        verdict = {
            'prediction': HARD_FALLBACK_LABEL,
            'confidence': 0.0,
            'vote_counts': {label: 0 for label in LABELS},
            'n_valid_votes': 0,
            'claim_outcomes': [],
            'applied_laws': [],
        }
        prediction_source = 'hard_fallback'

    # ==== DECIDER: nhan cuoi suy tu claim_outcomes, khong dung nhan model tu khai ====
    # Do tren lan chay truoc (mapper hoat dong, 2.3 claim/case, 33/50 case >=2 claim):
    #   holistic  strict .540 / macroF1 .508  |  derived  strict .640 / macroF1 .650
    #   nested CV 5-fold x200: holistic .445  |  derived .557  (base Qwen3.5-9B .405)
    #   hieu so cap derived-holistic: CI95 [.018, .266], P(derived > holistic) = .991
    # Quet 216 cau hinh nguong: 216/216 deu thang holistic (thap nhat .524) => ket luan
    # khong phu thuoc viec chon nguong. KHONG chinh nguong .85/.55/.30: chinh lam TE di
    # (chon nguong trong fold tut ve .526).
    holistic_label = verdict['prediction']
    derived_final = (
        derive_label(current['issue_map'], verdict['claim_outcomes'])
        if verdict['claim_outcomes'] else None
    )
    final_prediction = derived_final or holistic_label

    result = {
        'case_id': cid,
        'case_query': case['case_query'],
        'eval_seed': EVAL_SEED,
        'pipeline_version': PADO_PIPELINE_VERSION,
        'pipeline_contract_hash': PIPELINE_CONTRACT_HASH,
        'pipeline_stages': {
            'issue_mapper': current['issue_map'],
            'judge_votes': current['votes'],
        },
        'retrieved_laws': current['laws'],
        'raw_responses': current['raw'],
        'usage': aggregate_stage_usage(current['usage']),
        'duration_seconds': per_case_duration,
        'cache_key': cache_keys[cid],
        'prediction': final_prediction,
        'prediction_source': prediction_source,
        'decider': 'derived_from_claim_outcomes' if derived_final else 'holistic_fallback',
        'holistic_label': holistic_label,
        'derived_label_final': derived_final,
        'confidence': verdict['confidence'],
        'reasoning': (
            f"source={prediction_source}; votes={verdict['vote_counts']}; "
            f"valid_votes={verdict['n_valid_votes']}"
        ),
        'applied_laws': verdict['applied_laws'],
        'claim_outcomes': verdict['claim_outcomes'],
        'vote_counts': verdict['vote_counts'],
        'n_valid_votes': verdict['n_valid_votes'],
        # Giu de cham doi chung offline (holistic_label vs prediction).
        'derived_label_votes': [vote.get('derived_label') for vote in current['votes']],
        'mapper_fallback': bool(current['issue_map'].get('mapper_fallback')),
        'stage_errors': current['stage_errors'],
        'failed_stage': None,
        'error_kind': None,
        'error': None,
    }
    saved[cid] = result

    # Checkpoint từng case: nếu notebook bị ngắt sau đó, rerun không phải sinh lại case đã hoàn tất.
    atomic_write_json(result_path, saved)

# ==== TANG 1 (majority-at-claim): quyet dinh nhan cuoi tu SU DONG THUAN cac phieu ====
# Hau ky thuan tuy - chi doc judge_votes DA LUU, KHONG goi model. Nen chay lai voi cache
# khop (0 case trong cases_run) van cap nhat duoc nhan ma khong ton GPU. Ap cho MOI case
# trong saved, ke ca case nap tu checkpoint. prediction = derive(consensus) thay vi
# derive(phieu dai dien). Giu holistic_label + phieu dai dien de cham doi chung offline.
for _cid, _res in saved.items():
    _stages = _res.get('pipeline_stages') or {}
    _votes = _stages.get('judge_votes') or []
    _imap = _stages.get('issue_mapper') or {'claims': []}
    if not (_votes and _imap.get('claims')):
        continue
    _consensus = consensus_claim_outcomes(_votes, _imap)
    _label = derive_label(_imap, _consensus)
    if not _label:
        continue
    _res.setdefault('holistic_label', _res.get('prediction'))
    _res.setdefault('representative_claim_outcomes', _res.get('claim_outcomes'))
    _res['consensus_claim_outcomes'] = _consensus
    _res['claim_outcomes'] = _consensus
    _res['derived_label_final'] = _label
    _res['prediction'] = _label
    _res['decider'] = 'majority_at_claim_consensus'
atomic_write_json(result_path, saved)

expected_ids = {case['case_id'] for case in inference_cases}
missing_ids = sorted(expected_ids - set(saved))
invalid_ids = sorted(
    cid for cid in expected_ids if (saved.get(cid) or {}).get('prediction') not in LABELS
)
if missing_ids or invalid_ids:
    raise RuntimeError(f'Coverage invariant failed: missing={missing_ids}, invalid={invalid_ids}')
atomic_write_json(result_path, saved)
all_model_results = {MODEL: {EVAL_SEED: saved}}
mapper_fallbacks = sum(1 for v in saved.values() if v.get('mapper_fallback'))
claim_counts = {}
for value in saved.values():
    n = len((value.get('pipeline_stages') or {}).get('issue_mapper', {}).get('claims') or [])
    claim_counts[n] = claim_counts.get(n, 0) + 1
print(f'Issue mapper: fallback={mapper_fallbacks}/{len(saved)} | claims/case={dict(sorted(claim_counts.items()))}')
source_counts = {}
for value in saved.values():
    source = value.get('prediction_source', 'unknown')
    source_counts[source] = source_counts.get(source, 0) + 1
print(
    f'Completed v12: {sum(v.get("prediction") in LABELS for v in saved.values())}/'
    f'{len(inference_cases)} valid | sources={source_counts} | '
    f'{duration_all}s for {len(cases_run)} cases (~{per_case_duration}s/case).'
)


Benchmark manifest: 
outputs_alqac_e2e/v12_legal_pado_divisible_types_balanced_judge_private/benchmark_manifest_qwen3-5-9b.json
⠹ Running (1/1 containers active)... View app at 
=== v12 compact ensemble | seed=2026 | run 60/60 cases ===
⠸ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠇ Running (1/1 containers active)... View app at 
⠙ Running (1/1 containers active)... View app at 
⠼ Running (1/1 containers active)... View app at 
⠧ Running (1/1 containers active)... View app at 
⠋ Running (1/1 containers active)... View app at 
⠸ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 
⠋ Running (1/1 containers active)... View app at 
⠸ Running (1/1 containers active)... View app at 
⠦ Running (1/1 containers active)... View app at 
⠏ Running (1/1 containers active)... View app at 
⠹ Running (1/1 containers active)... View app at 
⠴ Running (1/1 containers active)... View app at 
⠏ Running (1/

In [8]:
# ==== PRIVATE TEST: KHONG co nhan gold => KHONG cham diem duoc ====
# Cell nay truoc day tinh accuracy/macro-F1/confusion matrix bang gold_by_case. Voi bai
# thi private, thu duy nhat kiem tra duoc la SUC KHOE pipeline: co case nao rong nhan,
# co bi cat output, mapper co chay khong, co phai dung fallback khong.
def health_report(model, eval_seed, result_map):
    rows = []
    for case in inference_cases:
        cid = case['case_id']
        item = result_map.get(cid, {})
        usage = item.get('usage') or {}
        stages = item.get('pipeline_stages') or {}
        rows.append({
            'case_id': cid,
            'prediction': item.get('prediction'),
            'is_valid_output': item.get('prediction') in LABELS,
            'decider': item.get('decider'),
            'prediction_source': item.get('prediction_source'),
            'n_claims': len((stages.get('issue_mapper') or {}).get('claims') or []),
            'mapper_fallback': bool(item.get('mapper_fallback', False)),
            'n_valid_votes': item.get('n_valid_votes'),
            'confidence': item.get('confidence'),
            'input_tokens': usage.get('input_tokens'),
            'output_tokens': usage.get('output_tokens'),
            'hit_max_new_tokens': bool(usage.get('hit_max_new_tokens', False)),
            'n_stage_errors': len(item.get('stage_errors') or []),
            'error_kind': item.get('error_kind'),
        })
    frame = pd.DataFrame(rows)
    n = len(frame)
    print(f'=== HEALTH {model} | seed={eval_seed} | {n} case ===')
    print('  nhan hop le            :', int(frame['is_valid_output'].sum()), '/', n)
    print('  mapper_fallback        :', int(frame['mapper_fallback'].sum()), '/', n)
    print('  bi cat output (truncate):', int(frame['hit_max_new_tokens'].sum()), '/', n)
    print('  case co loi stage      :', int((frame['n_stage_errors'] > 0).sum()), '/', n)
    print('  so claim/case          :', dict(sorted(frame['n_claims'].value_counts().items())))
    print('  nguon nhan             :', dict(frame['prediction_source'].value_counts()))
    print('  decider                :', dict(frame['decider'].value_counts(dropna=False)))
    print('  phan bo du doan        :', dict(frame['prediction'].value_counts(dropna=False)))
    invalid = frame.loc[~frame['is_valid_output'], 'case_id'].tolist()
    if invalid:
        print('  CANH BAO - case chua co nhan hop le:', invalid)
    frame.to_csv(OUTPUT_DIR / f'health_{slugify(model)}_seed-{eval_seed}.csv',
                 index=False, encoding='utf-8-sig')
    display(frame.head(10))
    return frame


for _model, _seed_results in all_model_results.items():
    for _seed, _results in _seed_results.items():
        health_report(_model, _seed, _results)


=== HEALTH Qwen3.5-9B | seed=2026 | 60 case ===
⠴ Running (1/1 containers active)... View app at 
  nhan hop le            : 60 / 60
⠴ Running (1/1 containers active)... View app at 
  mapper_fallback        : 2 / 60
⠴ Running (1/1 containers active)... View app at 
  bi cat output (truncate): 0 / 60
⠴ Running (1/1 containers active)... View app at 
  case co loi stage      : 4 / 60
⠴ Running (1/1 containers active)... View app at 
  so claim/case          : {1: 20, 2: 16, 3: 16, 4: 6, 5: 2}
⠴ Running (1/1 containers active)... View app at 
  nguon nhan             : {'judge_ensemble': np.int64(60)}
⠴ Running (1/1 containers active)... View app at 
  decider                : {'majority_at_claim_consensus': np.int64(60)}
⠴ Running (1/1 containers active)... View app at 
  phan bo du doan        : {'PARTIAL_A_WIN': np.int64(24), 'A_WIN': np.int64(17), 'B_WIN': np.int64(13), 
'PARTIAL_B_WIN': np.int64(6)}
⠴ Running (1/1 containers active)... View app at 
https://modal.com/apps/dangkhoaich

,case_id,prediction,is_valid_output,decider,prediction_source,n_claims,mapper_fallback,n_valid_votes,confidence,input_tokens,output_tokens,hit_max_new_tokens,n_stage_errors,error_kind
0,case_1615,A_WIN,True,majority_at_claim_consensus,judge_ensemble,1,True,2,0.950000,40909,23784,False,2,None
1,case_4110,A_WIN,True,majority_at_claim_consensus,judge_ensemble,1,False,3,0.916667,45218,25071,False,0,None
2,case_3487,A_WIN,True,majority_at_claim_consensus,judge_ensemble,3,False,3,0.935000,39394,22947,False,0,None
3,case_3199,A_WIN,True,majority_at_claim_consensus,judge_ensemble,1,False,3,0.900000,44935,23200,False,0,None
4,case_5000,A_WIN,True,majority_at_claim_consensus,judge_ensemble,1,False,3,0.906667,37845,22058,False,0,None
5,case_7377,A_WIN,True,majority_at_claim_consensus,judge_ensemble,1,False,3,0.865000,38617,20080,False,0,None
6,case_407,A_WIN,True,majority_at_claim_consensus,judge_ensemble,3,False,3,0.850000,38825,22815,False,0,None
7,case_6257,PARTIAL_A_WIN,True,majority_at_claim_consensus,judge_ensemble,2,False,3,0.850000,36830,24217,False,0,None
8,case_118,B_WIN,True,majority_at_claim_consensus,judge_ensemble,2,False,3,0.800000,44475,27887,False,0,None
9,case_1551,PARTIAL_A_WIN,True,majority_at_claim_consensus,judge_ensemble,1,False,3,0.783333,38891,24484,False,0,None


In [9]:
for model, seed_results in all_model_results.items():
    for eval_seed, results in seed_results.items():
        submission = []
        for case in inference_cases:
            item = results.get(case['case_id'], {})
            if item.get('prediction') not in LABELS:
                continue
            submission.append({
                'case_id': case['case_id'],
                'prediction': item['prediction'],
                'case_evidence': evidence_ids_by_case.get(case['case_id'], []),
                # law_evidence = TOAN BO dieu luat retrieval cua case (user chon 2026-07-24).
                # KHONG dung applied_laws: judge 9B gan nhu khong bao gio dien legal_basis_ranks
                # (do private run: 0/134 claim co rank) nen applied_laws rong 57/60 case.
                # Lay tu retrieval_cache => moi case du 14 dieu, khong phu thuoc judge.
                'law_evidence': [
                    {'law_id': law['law_id'], 'aid': int(law['aid'])}
                    for law in retrieval_cache.get(case['case_id'], [])
                ],
            })
        expected_ids = {case['case_id'] for case in inference_cases}
        submission_ids = [row['case_id'] for row in submission]
        if len(submission) != len(inference_cases) or set(submission_ids) != expected_ids:
            raise RuntimeError(
                f'Submission coverage failed: rows={len(submission)}, unique={len(set(submission_ids))}'
            )
        path = OUTPUT_DIR / f'submission_{slugify(model)}_seed-{eval_seed}.json'
        atomic_write_json(path, submission)
        n_failed = len(inference_cases) - len(submission)
        print(
            model,
            '| seed:', eval_seed,
            '| cases:', len(inference_cases),
            '| thieu nhan hop le:', n_failed,
            '| submission rows:', len(submission), '/', len(inference_cases),
            '|', path,
        )

print('Outputs:', OUTPUT_DIR.resolve())


Qwen3.5-9B | seed: 2026 | cases: 60 | thieu nhan hop le: 0 | submission rows: 60 / 60 | 
outputs_alqac_e2e/v12_legal_pado_divisible_types_balanced_judge_private/submission_qwen3-5-9b_seed-2026.json
⠦ Running (1/1 containers active)... View app at 
Outputs: /root/outputs_alqac_e2e/v12_legal_pado_divisible_types_balanced_judge_private
⠦ Running (1/1 containers active)... View app at 
https://modal.com/apps/dangkhoaichau/main/ap-8u5Ro5jtmrtw1fb8rE9QpL

In [10]:
# Dong Modal app: giai phong container GPU remote sau khi chay xong toan bo pipeline.
close_modal()
print('Da dong Modal app.')


⠦ Running (1/1 containers active)... View app at 
https://modal.com/apps/dangkhoaichau/main/ap-8u5Ro5jtmrtw1fb8rE9QpL
✓ App completed. View run at https://modal.com/apps/dangkhoaichau/main/ap-8u5Ro5jtmrtw1fb8rE9QpL
Da dong Modal app.
